In [0]:
# === EDA 1: IBGE — Agregados Básicos por Distritos (Censo 2022) ===
import pandas as pd
from zipfile import ZipFile

zf = ZipFile("/Volumes/workspace/bronze/raw/ibge/agregados_basicos.zip")
with zf.open("Agregados_por_distritos_basico_BR.csv") as f:
    df_ibge = pd.read_csv(f, encoding='latin-1', sep=';', na_values=['.'], decimal=',')

print("=" * 60)
print("IBGE — Agregados Básicos por Distritos (Censo 2022)")
print("=" * 60)
print(f"Shape: {df_ibge.shape[0]:,} linhas x {df_ibge.shape[1]} colunas\n")

print("Colunas e tipos:")
for c, t in df_ibge.dtypes.items():
    print(f"  {c:20s} {t}")

print(f"\nValores ausentes:")
miss = df_ibge.isnull().sum()
miss = miss[miss > 0]
if len(miss) > 0:
    for c, v in miss.items():
        print(f"  {c:20s} {v:>6,} ({v/len(df_ibge)*100:.1f}%)")
else:
    print("  Nenhum valor ausente")

print(f"\nCobertura geográfica:")
print(f"  Regiões:      {df_ibge['NM_REGIAO'].nunique()} — {', '.join(df_ibge['NM_REGIAO'].unique())}")
print(f"  UFs:          {df_ibge['NM_UF'].nunique()}")
print(f"  Municípios:   {df_ibge['NM_MUN'].nunique()}")
print(f"  Distritos:    {df_ibge['NM_DIST'].nunique()}")

print(f"\nEstatísticas descritivas (variáveis demográficas):")
num_cols = ['v0001','v0002','v0003','v0004','v0005','v0006','v0007','v0008','v0009','AREA_KM2']
print(df_ibge[num_cols].describe().round(2).to_string())

print(f"\nAmostra (primeiras 5 linhas, colunas selecionadas):")
show_cols = ['CD_DIST','NM_DIST','NM_UF','NM_MUN','AREA_KM2','v0001','v0002','v0005']
print(df_ibge[show_cols].head().to_string())

In [0]:
# === EDA 2: SSP — Dados Criminais do Estado de São Paulo ===
import os
import pandas as pd

ssp_dir = "/Volumes/workspace/bronze/raw/ssp"
ssp_files = sorted(f for f in os.listdir(ssp_dir) if f.endswith('.xlsx'))

print("=" * 60)
print("SSP — Dados Criminais do Estado de São Paulo")
print("=" * 60)
print(f"Arquivos disponíveis ({len(ssp_files)}):")
for f in ssp_files:
    size_mb = os.path.getsize(os.path.join(ssp_dir, f)) / (1024 * 1024)
    print(f"  {f:45s} {size_mb:>8.1f} MB")

In [0]:
%pip install openpyxl

# === EDA 2.1: SSP — Celulares Subtraídos (2026) ===
path_cel = os.path.join(ssp_dir, "CelularesSubtraidos_2026.xlsx")

# --- Descobrir abas disponíveis ---
xls_cel = pd.ExcelFile(path_cel)
print("=" * 60)
print("SSP — Celulares Subtraídos (2026)")
print("=" * 60)
print(f"Arquivo: {path_cel}")
print(f"Tamanho: {os.path.getsize(path_cel) / (1024 * 1024):.1f} MB")
print(f"\nAbas ({len(xls_cel.sheet_names)}):")
for s in xls_cel.sheet_names:
    print(f"  {s}")

# --- EDA detalhada: aba CELULAR_2026 ---
df_cel = pd.read_excel(path_cel, sheet_name="CELULAR_2026")
print(f"\n{'-' * 60}")
print(f"EDA — Aba CELULAR_2026")
print(f"{'-' * 60}")
print(f"Shape: {df_cel.shape[0]:,} linhas x {df_cel.shape[1]} colunas")

print(f"\nColunas e tipos:")
for c, t in df_cel.dtypes.items():
    print(f"  {c:35s} {t}")

print(f"\nValores ausentes:")
miss = df_cel.isnull().sum()
miss = miss[miss > 0]
if len(miss) > 0:
    for c, v in miss.items():
        print(f"  {c:35s} {v:>6,} ({v / len(df_cel) * 100:.1f}%)")
else:
    print("  Nenhum valor ausente")

# Identificar colunas temporais
col_data = [c for c in df_cel.columns if 'DATA' in c.upper() or 'ANO' in c.upper()]
if col_data:
    for c in col_data:
        print(f"\nPeríodo ({c}): {df_cel[c].min()} – {df_cel[c].max()}")

# Distribuição das principais variáveis categóricas
for c in df_cel.columns:
    if df_cel[c].dtype == "object" and df_cel[c].nunique() <= 30:
        print(f"\nDistribuição — {c}:")
        print(df_cel[c].value_counts().to_string())

# Estatísticas de variáveis numéricas
num_cols = df_cel.select_dtypes(include="number").columns.tolist()
if num_cols:
    print(f"\nEstatísticas descritivas (variáveis numéricas):")
    print(df_cel[num_cols].describe().round(2).to_string())

print(f"\nAmostra (primeiras 5 linhas):")
print(df_cel.head().to_string())

# Distribuição por CIDADE
if "CIDADE" in df_cel.columns:
    print(f"\n{'-' * 60}")
    print(f"Distribuição — CIDADE (top 20)")
    print(f"{'-' * 60}")
    print(df_cel["CIDADE"].value_counts().head(20).to_string())
    print(f"\nTotal de cidades: {df_cel['CIDADE'].nunique()}")
else:
    print("\nColuna CIDADE não encontrada no dataset.")

# Distribuição por BAIRRO
if "BAIRRO" in df_cel[df_cel["CIDADE"] == "S.PAULO"].columns:
    print(f"\n{'-' * 60}")
    print(f"Distribuição — BAIRRO (top 20)")
    print(f"{'-' * 60}")
    print(df_cel["BAIRRO"].value_counts().head(20).to_string())
    print(f"\nTotal de bairros: {df_cel['BAIRRO'].nunique()}")
else:
    print("\nColuna BAIRRO não encontrada no dataset.")




In [0]:
df_cel = df_cel[df_cel["CIDADE"] == "S.PAULO"]
print(df_cel["CIDADE"].unique())

In [0]:
# Distribuição por BAIRRO - São Paulo
df_cel = df_cel[df_cel["CIDADE"] == "S.PAULO"]
if "BAIRRO" in df_cel.columns:
    print(f"\n{'-' * 60}")
    print(f"Distribuição — BAIRRO (top 20)")
    print(f"{'-' * 60}")
    print(df_cel["BAIRRO"].value_counts().head(20).to_string())
    print(f"\nTotal de bairros: {df_cel['BAIRRO'].nunique()}")
else:
    print("\nColuna BAIRRO não encontrada no dataset.")

In [0]:
# === Visualização EDA 2.1: SSP — Celulares Subtraídos (São Paulo) ===
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

# ── Carregamento e filtro para S.PAULO ──────────────────────
ssp_dir = "/Volumes/workspace/bronze/raw/ssp"
path_cel = os.path.join(ssp_dir, "CelularesSubtraidos_2026.xlsx")
df_cel = pd.read_excel(path_cel, sheet_name="CELULAR_2026")
df_cel = df_cel[df_cel["CIDADE"] == "S.PAULO"].copy()

print(f"Registros em S.PAULO: {len(df_cel):,}")

# ── 1. Top 20 bairros com mais registros ─────────────────────
if "BAIRRO" in df_cel.columns:
    top_bairros = df_cel["BAIRRO"].value_counts().head(20)

    fig, ax = plt.subplots(figsize=(14, 8))
    ax.barh(top_bairros.index[::-1], top_bairros.values[::-1], color="steelblue")
    ax.set_xlabel("Quantidade de registros")
    ax.set_title("Top 20 Bairros — Celulares Subtraídos em S.PAULO", fontsize=13, pad=12)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    for i, v in enumerate(top_bairros.values[::-1]):
        ax.text(v + max(top_bairros.values) * 0.01, i, f"{v:,}", va="center", fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("Coluna BAIRRO não encontrada.")

# ── 2. Meses com mais registros de BO ───────────────────────
# Identificar coluna de data
col_data = [c for c in df_cel.columns if "DATA" in c.upper()]
if col_data:
    col = col_data[0]
    df_cel["_data"] = pd.to_datetime(df_cel[col], errors="coerce")
    df_cel["_mes"] = df_cel["_data"].dt.to_period("M")
    registros_mes = df_cel.dropna(subset=["_mes"]).groupby("_mes").size().sort_index()

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.bar(registros_mes.index.astype(str), registros_mes.values, color="crimson")
    ax.set_title(f"Registros de BO por Mês ({col})", fontsize=13, pad=12)
    ax.set_ylabel("Quantidade de registros")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    plt.xticks(rotation=45, ha="right")
    for i, v in enumerate(registros_mes.values):
        ax.text(i, v + max(registros_mes.values) * 0.01, f"{v:,}", ha="center", fontsize=8)
    plt.tight_layout()
    plt.show()

    # Tabela com top meses
    top_meses = registros_mes.sort_values(ascending=False).head(12)
    print(f"\nTop 12 meses com mais registros de BO:")
    for mes, qtd in top_meses.items():
        print(f"  {mes}  {qtd:>6,}")
else:
    print("Nenhuma coluna de DATA encontrada.")

# ── 3. Valores ausentes por variável ─────────────────────────
miss = df_cel.isnull().sum()
miss = miss[miss > 0].sort_values(ascending=True)
miss_pct = (miss / len(df_cel) * 100)

if len(miss) > 0:
    fig, ax = plt.subplots(figsize=(14, max(5, len(miss) * 0.4)))
    ax.barh(miss.index, miss_pct.values, color="#C44E52")
    ax.set_xlabel("% de valores ausentes")
    ax.set_title("Valores Ausentes por Variável (%) — Celulares Subtraídos S.PAULO", fontsize=13, pad=12)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}%"))
    for i, v in enumerate(miss_pct.values):
        ax.text(v + max(miss_pct.values) * 0.01, i, f"{v:.1f}%", va="center", fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum valor ausente encontrado.")

In [0]:
# === EDA 2.2: SSP — DadosProdutividade Policial (2026) ===
import os
import pandas as pd

ssp_dir = "/Volumes/workspace/bronze/raw/ssp"
path_dp = os.path.join(ssp_dir, "DadosProdutividade_2026.xlsx")

# ── 1. Abas disponíveis ──────────────────────────────────────
xls_dp = pd.ExcelFile(path_dp)
print("=" * 60)
print("SSP — DadosProdutividade Policial (2026)")
print("=" * 60)
print(f"Arquivo: {path_dp}")
print(f"Tamanho: {os.path.getsize(path_dp) / (1024 * 1024):.1f} MB")
print(f"\nAbas ({len(xls_dp.sheet_names)}):")
for s in xls_dp.sheet_names:
    print(f"  {s}")

abas_eda = [s for s in xls_dp.sheet_names if s != "Campos da Tabela_SP PROD"]
print(f"\nAbas para EDA ({len(abas_eda)}): {abas_eda}")

# ── 2. EDA por aba ──────────────────────────────────────────
for aba in abas_eda:
    df = pd.read_excel(path_dp, sheet_name=aba)
    print(f"\n{'#' * 60}")
    print(f"# EDA — Aba: {aba}")
    print(f"{'#' * 60}")
    print(f"Shape: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

    # (A) Tipagem de colunas
    print(f"\n--- A. Tipagem de colunas ---")
    for c, t in df.dtypes.items():
        print(f"  {c:40s} {t}")

    # (B) Taxa de null
    print(f"\n--- B. Taxa de valores nulos ---")
    miss = df.isnull().sum()
    miss = miss[miss > 0]
    if len(miss) > 0:
        for c, v in miss.items():
            pct = v / len(df) * 100
            print(f"  {c:40s} {v:>8,} ({pct:.2f}%)")
    else:
        print("  Nenhum valor nulo encontrado.")

    # (C) Quantidade de registros por bairro + % do total + taxa de flagrantes
    col_bairro = None
    for c in df.columns:
        if c.strip().upper() in ("BAIRRO", "NOME_BAIRRO", "BAIRRO_FATO", "DESC_BAIRRO"):
            col_bairro = c
            break
    if col_bairro is None:
        # busca parcial
        for c in df.columns:
            if "BAIRRO" in c.upper():
                col_bairro = c
                break

    col_flagrante = None
    for c in df.columns:
        if "FLAGRANTE" in c.upper():
            col_flagrante = c
            break

    print(f"\n--- C. Registros por bairro (top 10) ---")
    if col_bairro:
        total = len(df)
        contagem = df[col_bairro].value_counts().head(10)
        print(f"  Coluna de bairro utilizada: '{col_bairro}'")
        if col_flagrante:
            print(f"  Coluna de flagrante utilizada: '{col_flagrante}'")
            # Normalizar flagrante: contar onde o valor indica "Sim"/"S"/"True"/1
            flagrante_norm = df[col_flagrante].astype(str).str.strip().str.upper()
            eh_flagrante = flagrante_norm.isin(["SIM", "S", "TRUE", "1", "YES", "Y"])
            df_temp = df.copy()
            df_temp["_flag"] = eh_flagrante.astype(int)
            print(f"  {'Bairro':40s} {'Registros':>10s} {'% Total':>10s} {'% Flagrante':>12s}");
            print(f"  {'-'*40} {'-'*10} {'-'*10} {'-'*12}")
            for bairro, qtd in contagem.items():
                pct_total = qtd / total * 100
                flag_count = df_temp[df_temp[col_bairro] == bairro]["_flag"].sum()
                pct_flagrante = (flag_count / qtd * 100) if qtd > 0 else 0
                print(f"  {str(bairro):40s} {qtd:>10,} {pct_total:>9.2f}% {pct_flagrante:>11.2f}%")
        else:
            print(f"  (Coluna de flagrante não encontrada — exibindo apenas registros e % do total)")
            print(f"  {'Bairro':40s} {'Registros':>10s} {'% Total':>10s}");
            print(f"  {'-'*40} {'-'*10} {'-'*10}")
            for bairro, qtd in contagem.items():
                pct_total = qtd / total * 100
                print(f"  {str(bairro):40s} {qtd:>10,} {pct_total:>9.2f}%")
    else:
        print("  Coluna de BAIRRO não encontrada nesta aba.")

    print(f"\n{'#' * 60}\n")

In [0]:
# === Visualização EDA 2.2: SSP — DadosProdutividade Policial (2026) ===
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

ssp_dir = "/Volumes/workspace/bronze/raw/ssp"
path_dp = os.path.join(ssp_dir, "DadosProdutividade_2026.xlsx")

xls_dp = pd.ExcelFile(path_dp)
abas = [s for s in xls_dp.sheet_names if s != "Campos da Tabela_SP PROD"]
print(f"Abas para visualização: {abas}")

for aba in abas:
    df = pd.read_excel(path_dp, sheet_name=aba)
    print(f"\n{'#' * 70}")
    print(f"# Visualizações — Aba: {aba}  ({df.shape[0]:,} linhas x {df.shape[1]} colunas)")
    print(f"{'#' * 70}")

    # ── 0. Valores ausentes por variável ─────────────────────
    miss = df.isnull().sum()
    miss = miss[miss > 0].sort_values(ascending=True)
    miss_pct = (miss / len(df) * 100)

    if len(miss) > 0:
        fig, ax = plt.subplots(figsize=(14, max(4, len(miss) * 0.4)))
        ax.barh(miss.index, miss_pct.values, color="#C44E52")
        ax.set_xlabel("% de valores ausentes")
        ax.set_title(f"Valores Ausentes (%) — {aba}", fontsize=13, pad=12)
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}%"))
        for i, v in enumerate(miss_pct.values):
            ax.text(v + max(miss_pct.values) * 0.01, i, f"{v:.1f}%", va="center", fontsize=8)
        plt.tight_layout()
        plt.show()
    else:
        print("  Nenhum valor ausente.")

    # ── Identificar coluna de bairro ────────────────────────
    col_bairro = None
    for c in df.columns:
        if c.strip().upper() in ("BAIRRO", "NOME_BAIRRO", "BAIRRO_FATO", "DESC_BAIRRO"):
            col_bairro = c
            break
    if col_bairro is None:
        for c in df.columns:
            if "BAIRRO" in c.upper():
                col_bairro = c
                break

    # ── Identificar coluna de flagrante ─────────────────────
    col_flagrante = None
    for c in df.columns:
        if "FLAGRANTE" in c.upper():
            col_flagrante = c
            break

    # ── 1. Top 10 bairros por quantidade de registros ───────
    if col_bairro:
        top_bairros = df[col_bairro].value_counts().head(10)
        total = len(df)

        fig, ax = plt.subplots(figsize=(14, 8))
        bars = ax.barh(top_bairros.index[::-1], top_bairros.values[::-1], color="steelblue")
        ax.set_xlabel("Quantidade de registros")
        ax.set_title(f"Top 10 Bairros — {aba}", fontsize=13, pad=12)
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
        for i, v in enumerate(top_bairros.values[::-1]):
            pct = v / total * 100
            ax.text(v + max(top_bairros.values) * 0.01, i, f"{v:,} ({pct:.1f}%)", va="center", fontsize=8)
        plt.tight_layout()
        plt.show()

        # ── 2. Taxa de flagrante por bairro (10) ──────────
        if col_flagrante:
            flagrante_norm = df[col_flagrante].astype(str).str.strip().str.upper()
            eh_flagrante = flagrante_norm.isin(["SIM", "S", "TRUE", "1", "YES", "Y"])
            df_temp = df.copy()
            df_temp["_flag"] = eh_flagrante.astype(int)

            bairro_stats = (
                df_temp.groupby(col_bairro)
                .agg(registros=(col_bairro, "size"), flagrantes=("_flag", "sum"))
            )
            bairro_stats["pct_flagrante"] = bairro_stats["flagrantes"] / bairro_stats["registros"] * 100
            top_flag = bairro_stats.sort_values("registros", ascending=False).head(10)

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

            # Registros por bairro
            ax1.barh(top_flag.index[::-1], top_flag["registros"].values[::-1], color="steelblue")
            ax1.set_xlabel("Quantidade de registros")
            ax1.set_title(f"Registros por Bairro — {aba}", fontsize=12, pad=10)
            ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

            # % flagrante por bairro
            ax2.barh(top_flag.index[::-1], top_flag["pct_flagrante"].values[::-1], color="crimson")
            ax2.set_xlabel("% Flagrante")
            ax2.set_title(f"Taxa de Flagrante por Bairro — {aba}", fontsize=12, pad=10)
            ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}%"))
            for i, v in enumerate(top_flag["pct_flagrante"].values[::-1]):
                ax2.text(v + max(top_flag["pct_flagrante"].values) * 0.01, i, f"{v:.1f}%", va="center", fontsize=8)

            plt.suptitle(f"Análise por Bairro — {aba}", fontsize=14, y=1.02)
            plt.tight_layout()
            plt.show()

            # ── 3. Proporção flagrante vs não-flagrante (geral) ─
            flag_counts = df_temp["_flag"].value_counts()
            labels = ["Sem Flagrante", "Com Flagrante"]
            values = [flag_counts.get(0, 0), flag_counts.get(1, 0)]
            colors = ["#4C72B0", "#C44E52"]

            fig, ax = plt.subplots(figsize=(8, 6))
            ax.pie(values, labels=labels, autopct="%1.1f%%", colors=colors, startangle=90)
            ax.set_title(f"Proporção Flagrante vs Não-Flagrante — {aba}", fontsize=13, pad=12)
            plt.tight_layout()
            plt.show()
        else:
            print(f"  Coluna de flagrante não encontrada na aba {aba}.")
    else:
        print(f"  Coluna de BAIRRO não encontrada na aba {aba}.")

    print(f"\n{'#' * 70}\n")

In [0]:
# EDA detalhada do MDIP
print(f"\n--- EDA detalhada: MDIP_2026.xlsx ---")
df_ssp = pd.read_excel(os.path.join(ssp_dir, "MDIP_2026.xlsx"), sheet_name="MDIP_2013_A_2026")
print(f"Shape: {df_ssp.shape[0]:,} linhas x {df_ssp.shape[1]} colunas")

print(f"\nColunas e tipos:")
for c, t in df_ssp.dtypes.items():
    print(f"  {c:30s} {t}")

print(f"\nValores ausentes:")
miss = df_ssp.isnull().sum()
miss = miss[miss > 0]
for c, v in miss.items():
    print(f"  {c:30s} {v:>6,} ({v/len(df_ssp)*100:.1f}%)")

print(f"\nPeríodo: {df_ssp['ANO ESTATISTICA'].min()}–{df_ssp['ANO ESTATISTICA'].max()}")

print(f"\nNatureza apurada (top 10):")
print(df_ssp['NATUREZA_APURADA'].value_counts().head(10).to_string())

print(f"\nSexo das pessoas:")
print(df_ssp['SEXO_PESSOA'].value_counts().to_string())

print(f"\nCor de pele:")
print(df_ssp['COR_PELE'].value_counts().to_string())

print(f"\nTipo de local (top 10):")
print(df_ssp['DESC_TIPOLOCAL'].value_counts().head(10).to_string())

print(f"\nEstatísticas de idade:")
print(df_ssp['IDADE_PESSOA'].describe().to_string())

print(f"\nAmostra (primeiras 3 linhas, colunas selecionadas):")
show_cols = ['MUNICIPIO_CIRCUNSCRICAO','ANO ESTATISTICA','DATA_FATO','DESC_TIPOLOCAL','NATUREZA_APURADA','SEXO_PESSOA','IDADE_PESSOA','COR_PELE']
print(df_ssp[show_cols].head(3).to_string())

In [0]:
# === EDA 2: SSP — Visualizações das estatísticas ===
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

ssp_dir = "/Volumes/workspace/bronze/raw/ssp"
df_ssp = pd.read_excel(
    os.path.join(ssp_dir, "MDIP_2026.xlsx"),
    sheet_name="MDIP_2013_A_2026",
)

# ── 1. Tamanho dos arquivos da SSP ──────────────────────────
ssp_files = sorted(f for f in os.listdir(ssp_dir) if f.endswith(".xlsx"))
sizes = [
    os.path.getsize(os.path.join(ssp_dir, f)) / (1024 * 1024)
    for f in ssp_files
]

fig, ax = plt.subplots()
ax.barh(ssp_files, sizes, color="steelblue")
ax.set_xlabel("Tamanho (MB)")
ax.set_title("Tamanho dos arquivos SSP (MB)")
ax.invert_yaxis()
for i, s in enumerate(sizes):
    ax.text(s + 0.1, i, f"{s:.1f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

# ── 2. Natureza apurada (top 10) ──────────────────────────────
top_nat = df_ssp["NATUREZA_APURADA"].value_counts().head(10)

fig, ax = plt.subplots()
ax.barh(top_nat.index[::-1], top_nat.values[::-1], color="crimson")
ax.set_xlabel("Quantidade de registros")
ax.set_title("Top 10 — Natureza Apurada")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
for i, v in enumerate(top_nat.values[::-1]):
    ax.text(v + max(top_nat.values) * 0.01, i, f"{v:,}", va="center", fontsize=8)
plt.tight_layout()
plt.show()

# ── 3. Sexo das pessoas ──────────────────────────────────────
sexo = df_ssp["SEXO_PESSOA"].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

cores_sexo = ["#4C72B0", "#DD8452", "#937860", "#8172D3"]
ax1.bar(sexo.index, sexo.values, color=cores_sexo[: len(sexo)])
ax1.set_title("Sexo das Pessoas — Barras")
ax1.set_ylabel("Quantidade")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
for i, v in enumerate(sexo.values):
    ax1.text(i, v + max(sexo.values) * 0.01, f"{v:,}", ha="center", fontsize=9)

ax2.pie(
    sexo.values,
    labels=sexo.index,
    autopct="%1.1f%%",
    colors=cores_sexo[: len(sexo)],
    startangle=90,
)
ax2.set_title("Sexo das Pessoas — Proporção")
plt.tight_layout()
plt.show()

# ── 4. Cor de pele ───────────────────────────────────────────
cor_pele = df_ssp["COR_PELE"].value_counts()

fig, ax = plt.subplots()
ax.bar(cor_pele.index, cor_pele.values, color="#55A868")
ax.set_title("Cor de Pele — Distribuição")
ax.set_ylabel("Quantidade")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.xticks(rotation=30, ha="right")
for i, v in enumerate(cor_pele.values):
    ax.text(i, v + max(cor_pele.values) * 0.01, f"{v:,}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

# ── 5. Tipo de local (top 10) ───────────────────────────────
top_local = df_ssp["DESC_TIPOLOCAL"].value_counts().head(10)

fig, ax = plt.subplots()
ax.barh(top_local.index[::-1], top_local.values[::-1], color="#8172D3")
ax.set_xlabel("Quantidade de registros")
ax.set_title("Top 10 — Tipo de Local")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
for i, v in enumerate(top_local.values[::-1]):
    ax.text(v + max(top_local.values) * 0.01, i, f"{v:,}", va="center", fontsize=8)
plt.tight_layout()
plt.show()

# ── 6. Distribuição de idade ────────────────────────────────
idade = pd.to_numeric(df_ssp["IDADE_PESSOA"], errors="coerce").dropna()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(idade, bins=40, color="#C44E52", edgecolor="white", alpha=0.8)
ax1.set_title("Distribuição de Idade — Histograma")
ax1.set_xlabel("Idade")
ax1.set_ylabel("Frequência")
ax1.axvline(idade.mean(), color="black", linestyle="--", linewidth=1.5, label=f"Média = {idade.mean():.1f}")
ax1.axvline(idade.median(), color="blue", linestyle="--", linewidth=1.5, label=f"Mediana = {idade.median():.1f}")
ax1.legend(fontsize=9)

bp = ax2.boxplot(idade, vert=True, patch_artist=True, showfliers=True)
bp["boxes"][0].set_facecolor("#C44E52")
ax2.set_title("Distribuição de Idade — Boxplot")
ax2.set_ylabel("Idade")
ax2.set_xticks([])

stats_txt = (
    f"min  = {idade.min():.0f}\n"
    f"Q1   = {idade.quantile(.25):.0f}\n"
    f"med  = {idade.median():.0f}\n"
    f"Q3   = {idade.quantile(.75):.0f}\n"
    f"max  = {idade.max():.0f}"
)
ax2.text(1.35, idade.median(), stats_txt, fontsize=9, va="center", family="monospace")

plt.tight_layout()
plt.show()

# ── 7. Valores ausentes (%) por variável ────────────────────
miss = df_ssp.isnull().sum()
miss = miss[miss > 0].sort_values(ascending=True)
miss_pct = (miss / len(df_ssp) * 100)

if len(miss) > 0:
    fig, ax = plt.subplots()
    ax.barh(miss.index, miss_pct.values, color="crimson")
    ax.set_xlabel("% de valores ausentes")
    ax.set_title("Valores Ausentes por Variável (%)")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}%"))
    for i, v in enumerate(miss_pct.values):
        ax.text(v + max(miss_pct.values) * 0.01, i, f"{v:.1f}%", va="center", fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum valor ausente encontrado.")

# ── 8. Registros por ano ────────────────────────────────────
por_ano = df_ssp["ANO ESTATISTICA"].value_counts().sort_index()

fig, ax = plt.subplots()
ax.bar(por_ano.index.astype(str), por_ano.values, color="#4C72B0")
ax.set_title("Quantidade de Registros por Ano")
ax.set_ylabel("Quantidade")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.xticks(rotation=45)
for i, v in enumerate(por_ano.values):
    ax.text(i, v + max(por_ano.values) * 0.01, f"{v:,}", ha="center", fontsize=8)
plt.tight_layout()
plt.show()

In [0]:
# === EDA: SSP — ObjetosSubtraidos (2026) ===
import os
import pandas as pd

ssp_dir = "/Volumes/workspace/bronze/raw/ssp"
path_os = os.path.join(ssp_dir, "ObjetosSubtraidos_2026.xlsx")

# ── 1. Abas disponíveis ──────────────────────────────────────
xls_os = pd.ExcelFile(path_os)
print("=" * 60)
print("SSP — ObjetosSubtraidos (2026)")
print("=" * 60)
print(f"Arquivo: {path_os}")
print(f"Tamanho: {os.path.getsize(path_os) / (1024 * 1024):.1f} MB")
print(f"\nAbas ({len(xls_os.sheet_names)}):")
for s in xls_os.sheet_names:
    print(f"  {s}")

# Filtrar abas de EDA (excluir METODOLOGIA e DICIONARIO)
abas_eda = [
    s for s in xls_os.sheet_names
    if not any(k in s.upper() for k in ("METODOLOG", "DICIONARIO", "DICCIONARIO", "DIC"))
]
print(f"\nAbas para EDA ({len(abas_eda)}): {abas_eda}")

# ── 2. EDA por aba ──────────────────────────────────────────
for aba in abas_eda:
    df = pd.read_excel(path_os, sheet_name=aba)
    print(f"\n{'#' * 60}")
    print(f"# EDA — Aba: {aba}")
    print(f"{'#' * 60}")
    print(f"Shape: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

    # (A) Tipagem de colunas
    print(f"\n{'─' * 40}")
    print("(A) Tipagem de colunas")
    print(f"{'─' * 40}")
    for c, t in df.dtypes.items():
        print(f"  {c:40s} {t}")

    # (B) Taxa de null por coluna
    print(f"\n{'─' * 40}")
    print("(B) Taxa de valores nulos")
    print(f"{'─' * 40}")
    miss = df.isnull().sum()
    miss_pct = (miss / len(df) * 100)
    miss_df = pd.DataFrame({
        "nulos": miss,
        "taxa_null_%": miss_pct.round(2),
    }).sort_values("taxa_null_%", ascending=False)
    has_null = miss_df[miss_df["nulos"] > 0]
    if len(has_null) > 0:
        for c, row in has_null.iterrows():
            print(f"  {c:40s} {row['nulos']:>8,}  ({row['taxa_null_%']:.2f}%)")
    else:
        print("  Nenhum valor nulo encontrado.")

    # (C) Quantidade de registros por bairro + % do total
    print(f"\n{'─' * 40}")
    print("(C) Registros por bairro (% do total)")
    print(f"{'─' * 40}")

    # Identificar coluna de bairro
    col_bairro = None
    for c in df.columns:
        if c.strip().upper() in ("BAIRRO", "NOME_BAIRRO", "BAIRRO_FATO", "DESC_BAIRRO"):
            col_bairro = c
            break
    if col_bairro is None:
        for c in df.columns:
            if "BAIRRO" in c.upper():
                col_bairro = c
                break

    if col_bairro:
        total = len(df)
        bairro_counts = df[col_bairro].value_counts()
        bairro_pct = (bairro_counts / total * 100).round(2)
        bairro_df = pd.DataFrame({
            "registros": bairro_counts,
            "%_do_total": bairro_pct,
        })
        print(f"  Coluna identificada: '{col_bairro}'")
        print(f"  Total de bairros únicos: {bairro_counts.shape[0]}")
        print(f"  Total de registros: {total:,}\n")
        print(f"  --- Top 10 bairros ---")
        print(f"  {'Bairro':<40s} {'Registros':>10s} {'% Total':>10s}")
        print(f"  {'-' * 40} {'-' * 10} {'-' * 10}")
        for bairro, row in bairro_df.head(10).iterrows():
            print(f"  {str(bairro):<40s} {row['registros']:>10,} {row['%_do_total']:>9.2f}%")
        print(f"  \n  Total Top 10:   {bairro_df.head(10)['registros'].sum():>10,}  ({bairro_df.head(10)['%_do_total'].sum():.2f}% do total)")
    else:
        print(f"  Coluna de BAIRRO não encontrada na aba {aba}.")
        print(f"  Colunas disponíveis: {list(df.columns)}")

    print(f"\n{'#' * 60}\n")

In [0]:
# === Visualização EDA 2.4: SSP — ObjetosSubtraidos (2026) ===
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

ssp_dir = "/Volumes/workspace/bronze/raw/ssp"
path_os = os.path.join(ssp_dir, "ObjetosSubtraidos_2026.xlsx")

xls_os = pd.ExcelFile(path_os)

# Filtrar abas de EDA (excluir METODOLOGIA e DICIONARIO)
abas_eda = [
    s for s in xls_os.sheet_names
    if not any(k in s.upper() for k in ("METODOLOG", "DICIONARIO", "DICCIONARIO", "DIC"))
]
print(f"Abas para visualização: {abas_eda}")

for aba in abas_eda:
    df = pd.read_excel(path_os, sheet_name=aba)
    print(f"\n{'#' * 70}")
    print(f"# Visualizações — Aba: {aba}  ({df.shape[0]:,} linhas x {df.shape[1]} colunas)")
    print(f"{'#' * 70}")

    # ── 0. Valores ausentes por variável ─────────────────────
    miss = df.isnull().sum()
    miss = miss[miss > 0].sort_values(ascending=True)
    miss_pct = (miss / len(df) * 100)

    if len(miss) > 0:
        fig, ax = plt.subplots(figsize=(14, max(4, len(miss) * 0.4)))
        ax.barh(miss.index, miss_pct.values, color="#C44E52")
        ax.set_xlabel("% de valores ausentes")
        ax.set_title(f"Valores Ausentes (%) — {aba}", fontsize=13, pad=12)
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}%"))
        for i, v in enumerate(miss_pct.values):
            ax.text(v + max(miss_pct.values) * 0.01, i, f"{v:.1f}%", va="center", fontsize=8)
        plt.tight_layout()
        plt.show()
    else:
        print("  Nenhum valor ausente.")

    # ── Identificar coluna de bairro ────────────────────────
    col_bairro = None
    for c in df.columns:
        if c.strip().upper() in ("BAIRRO", "NOME_BAIRRO", "BAIRRO_FATO", "DESC_BAIRRO"):
            col_bairro = c
            break
    if col_bairro is None:
        for c in df.columns:
            if "BAIRRO" in c.upper():
                col_bairro = c
                break

    # ── Identificar coluna de quantidade (objeto subtraído) ──
    col_qtd = None
    for c in df.columns:
        if c.strip().upper() in ("QTDE", "QTD", "QUANTIDADE", "QUANT_OBJETOS", "QTDE_OBJETOS"):
            col_qtd = c
            break
    if col_qtd is None:
        for c in df.columns:
            if any(k in c.upper() for k in ("QTD", "QUANT")) and df[c].dtype in ("int64", "float64", "Int64"):
                col_qtd = c
            break

    # ── 1. Top 10 bairros por quantidade de registros ───────
    if col_bairro:
        top_bairros = df[col_bairro].value_counts().head(10)
        total = len(df)

        fig, ax = plt.subplots(figsize=(14, 8))
        bars = ax.barh(top_bairros.index[::-1], top_bairros.values[::-1], color="steelblue")
        ax.set_xlabel("Quantidade de registros")
        ax.set_title(f"Top 10 Bairros — {aba}", fontsize=13, pad=12)
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
        for i, v in enumerate(top_bairros.values[::-1]):
            pct = v / total * 100
            ax.text(v + max(top_bairros.values) * 0.01, i, f"{v:,} ({pct:.1f}%)", va="center", fontsize=8)
        plt.tight_layout()
        plt.show()

        # ── 2. Top 10 bairros por quantidade de objetos subtraídos ──
        if col_qtd:
            qtd_por_bairro = df.groupby(col_bairro)[col_qtd].sum().sort_values(ascending=False).head(10)
            total_qtd = df[col_qtd].sum()

            fig, ax = plt.subplots(figsize=(14, 8))
            ax.barh(qtd_por_bairro.index[::-1], qtd_por_bairro.values[::-1], color="#C44E52")
            ax.set_xlabel(f"Total de {col_qtd}")
            ax.set_title(f"Top 10 Bairros — {col_qtd} — {aba}", fontsize=13, pad=12)
            ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
            for i, v in enumerate(qtd_por_bairro.values[::-1]):
                pct = v / total_qtd * 100 if total_qtd > 0 else 0
                ax.text(v + max(qtd_por_bairro.values) * 0.01, i, f"{v:,.0f} ({pct:.1f}%)", va="center", fontsize=8)
            plt.tight_layout()
            plt.show()

        # ── 3. Distribuição cumulativa dos bairros (Pareto) ────
        all_bairros = df[col_bairro].value_counts()
        cumul_pct = all_bairros.cumsum() / all_bairros.sum() * 100

        fig, ax = plt.subplots(figsize=(14, 5))
        ax.bar(range(min(30, len(all_bairros))), all_bairros.values[:30], color="steelblue", alpha=0.8)
        ax.set_xlabel("Rank do bairro (top 30)")
        ax.set_ylabel("Quantidade de registros")
        ax.set_title(f"Distribuição de Registros por Bairro (Pareto) — {aba}", fontsize=13, pad=12)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

        ax2 = ax.twinx()
        ax2.plot(range(min(30, len(cumul_pct))), cumul_pct.values[:30], color="crimson", marker="o", linewidth=2)
        ax2.set_ylabel("% acumulada", color="crimson")
        ax2.set_ylim(0, 105)
        ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
        ax2.tick_params(axis="y", labelcolor="crimson")
        plt.tight_layout()
        plt.show()

    else:
        print(f"  Coluna de BAIRRO não encontrada. Colunas: {list(df.columns)}")

    print()

# ── Resumo consolidado: Top 10 bairros por aba (tabela) ────
print(f"\n{'=' * 70}")
print("RESUMO CONSOLIDADO — Top 10 Bairros por Aba")
print(f"{'=' * 70}")
for aba in abas_eda:
    df = pd.read_excel(path_os, sheet_name=aba)
    col_bairro = None
    for c in df.columns:
        if c.strip().upper() in ("BAIRRO", "NOME_BAIRRO", "BAIRRO_FATO", "DESC_BAIRRO"):
            col_bairro = c
            break
    if col_bairro is None:
        for c in df.columns:
            if "BAIRRO" in c.upper():
                col_bairro = c
                break
    if col_bairro:
        top10 = df[col_bairro].value_counts().head(10)
        total = len(df)
        print(f"\n─ Aba: {aba} ({total:,} registros, {df[col_bairro].nunique()} bairros únicos) ─")
        print(f"  {'Bairro':<45s} {'Registros':>10s} {'% Total':>8s}")
        print(f"  {'-' * 45} {'-' * 10} {'-' * 8}")
        for bairro, cnt in top10.items():
            print(f"  {str(bairro):<45s} {cnt:>10,} {cnt/total*100:>7.1f}%")
        print(f"  {'Top 10':<45s} {top10.sum():>10,} {top10.sum()/total*100:>7.1f}%")
    else:
        print(f"\n─ Aba: {aba} — coluna de bairro não encontrada ─")

In [0]:
# === EDA: SSP — SPDadosCriminais_2026 — Unificação das 2 últimas abas + EDA (S.PAULO por bairros) ===
%pip install openpyxl
import os
import pandas as pd

ssp_dir = "/Volumes/workspace/bronze/raw/ssp"
path_sp = os.path.join(ssp_dir, "SPDadosCriminais_2026.xlsx")

# ── 1. Carregar o arquivo e listar abas ───────────────────────
xls_sp = pd.ExcelFile(path_sp)
print("=" * 70)
print("SSP — SPDadosCriminais (2026)")
print("=" * 70)
print(f"Arquivo: {path_sp}")
print(f"Tamanho: {os.path.getsize(path_sp) / (1024 * 1024):.1f} MB")
print(f"\nAbas ({len(xls_sp.sheet_names)}):")
for i, s in enumerate(xls_sp.sheet_names):
    print(f"  [{i}] {s}")

# ── 2. Unificar as duas últimas abas em um único DataFrame ────
ultimas_duas = xls_sp.sheet_names[-2:]
print(f"\nUnificando as duas últimas abas: {ultimas_duas}")

dfs = []
for aba in ultimas_duas:
    df_aba = pd.read_excel(path_sp, sheet_name=aba)
    print(f"  {aba}: {df_aba.shape[0]:,} linhas x {df_aba.shape[1]} colunas")
    dfs.append(df_aba)

df_unificado = pd.concat(dfs, ignore_index=True, sort=False)
print(f"\nDataFrame unificado: {df_unificado.shape[0]:,} linhas x {df_unificado.shape[1]} colunas")

# ── 3. Identificar coluna de cidade ──────────────────────────
col_cidade = None
for c in df_unificado.columns:
    if c.strip().upper() in ("CIDADE", "MUNICIPIO", "MUNICÍPIO", "CIDADE_FATO", "NOME_MUNICIPIO", "NOME_MUNICÍPIO"):
        col_cidade = c
        break
if col_cidade is None:
    for c in df_unificado.columns:
        if "CIDADE" in c.upper() or "MUNICI" in c.upper():
            col_cidade = c
            break

print(f"\nColuna de cidade identificada: '{col_cidade}'")

# ── 4. Filtrar para S.PAULO ───────────────────────────────────
if col_cidade:
    # Exibir amostra de valores únicos de cidade para conferência
    print(f"\nAmostra de cidades (top 15 por frequência):")
    print(df_unificado[col_cidade].value_counts().head(15).to_string())

    # Filtrar S.PAULO (busca flexível para variações de casing/acentuação)
    mask_sp = df_unificado[col_cidade].astype(str).str.strip().str.upper() == "S.PAULO"
    df_sp = df_unificado[mask_sp].copy()
    print(f"\nRegistros para S.PAULO: {df_sp.shape[0]:,} linhas x {df_sp.shape[1]} colunas")
else:
    print("\n⚠ Coluna de cidade não encontrada. Usando DataFrame completo.")
    df_sp = df_unificado.copy()

# ── 5. EDA — Visão geral do DataFrame filtrado ───────────────
print(f"\n{'#' * 70}")
print(f"# EDA — S.PAULO")
print(f"{'#' * 70}")
print(f"Shape: {df_sp.shape[0]:,} linhas x {df_sp.shape[1]} colunas")

# (A) Tipagem de colunas
print(f"\n{'─' * 50}")
print("(A) Tipagem de colunas")
print(f"{'─' * 50}")
for c, t in df_sp.dtypes.items():
    print(f"  {c:40s} {t}")

# (B) Valores ausentes por coluna
print(f"\n{'─' * 50}")
print("(B) Taxa de valores nulos")
print(f"{'─' * 50}")
miss = df_sp.isnull().sum()
miss_pct = (miss / len(df_sp) * 100).round(2)
miss_df = pd.DataFrame({"nulos": miss, "taxa_null_%": miss_pct}).sort_values("taxa_null_%", ascending=False)
has_null = miss_df[miss_df["nulos"] > 0]
if len(has_null) > 0:
    for c, row in has_null.iterrows():
        print(f"  {c:40s} {row['nulos']:>8,}  ({row['taxa_null_%']:.2f}%)")
else:
    print("  Nenhum valor nulo encontrado.")

# (C) Estatísticas descritivas de colunas numéricas
print(f"\n{'─' * 50}")
print("(C) Estatísticas descritivas (colunas numéricas)")
print(f"{'─' * 50}")
num_cols = df_sp.select_dtypes(include=["number"]).columns
if len(num_cols) > 0:
    print(df_sp[num_cols].describe().round(2).to_string())
else:
    print("  Nenhuma coluna numérica encontrada.")

# ── 6. EDA orientado por bairros ─────────────────────────────
print(f"\n{'─' * 50}")
print("(D) Análise por bairros")
print(f"{'─' * 50}")

# Identificar coluna de bairro
col_bairro = None
for c in df_sp.columns:
    if c.strip().upper() in ("BAIRRO", "NOME_BAIRRO", "BAIRRO_FATO", "DESC_BAIRRO"):
        col_bairro = c
        break
if col_bairro is None:
    for c in df_sp.columns:
        if "BAIRRO" in c.upper():
            col_bairro = c
            break

if col_bairro:
    total = len(df_sp)
    bairro_counts = df_sp[col_bairro].value_counts()
    bairro_pct = (bairro_counts / total * 100).round(2)
    bairro_df = pd.DataFrame({"registros": bairro_counts, "%_do_total": bairro_pct})

    print(f"  Coluna identificada: '{col_bairro}'")
    print(f"  Total de bairros únicos: {bairro_counts.shape[0]}")
    print(f"  Total de registros (S.PAULO): {total:,}")

    # D1. Top 20 bairros por quantidade de registros
    print(f"\n  --- Top 20 bairros por registros ---")
    print(f"  {'Bairro':<45s} {'Registros':>10s} {'% Total':>10s}")
    print(f"  {'-' * 45} {'-' * 10} {'-' * 10}")
    for bairro, row in bairro_df.head(20).iterrows():
        print(f"  {str(bairro):<45s} {row['registros']:>10,} {row['%_do_total']:>9.2f}%")
    print(f"  \n  Total Top 20:   {bairro_df.head(20)['registros'].sum():>10,}  ({bairro_df.head(20)['%_do_total'].sum():.2f}% do total)")

    # D2. Bottom 20 bairros (menos registros)
    print(f"\n  --- Bottom 20 bairros por registros ---")
    print(f"  {'Bairro':<45s} {'Registros':>10s} {'% Total':>10s}")
    print(f"  {'-' * 45} {'-' * 10} {'-' * 10}")
    for bairro, row in bairro_df.tail(20).iterrows():
        print(f"  {str(bairro):<45s} {row['registros']:>10,} {row['%_do_total']:>9.2f}%")

    # D3. Bairros com registros nulos ou vazios
    nulos_bairro = df_sp[col_bairro].isnull().sum()
    vazios_bairro = (df_sp[col_bairro].astype(str).str.strip() == "").sum()
    print(f"\n  Registros com bairro nulo: {nulos_bairro:,} ({nulos_bairro/total*100:.2f}%)")
    print(f"  Registros com bairro vazio: {vazios_bairro:,} ({vazios_bairro/total*100:.2f}%)")

    # D4. Distribuição de registros por bairro (estatísticas)
    print(f"\n  --- Estatísticas da distribuição de registros por bairro ---")
    print(f"  Média de registros por bairro:    {bairro_counts.mean():,.1f}")
    print(f"  Mediana de registros por bairro: {bairro_counts.median():,.1f}")
    print(f"  Desvio padrão:                   {bairro_counts.std():,.1f}")
    print(f"  Mínimo:                          {bairro_counts.min():,}")
    print(f"  Máximo:                          {bairro_counts.max():,}")

    # D5. Curva de Pareto — concentração nos top N bairros
    print(f"\n  --- Concentração acumulada (Pareto) ---")
    cumul = bairro_pct.cumsum()
    for n in [5, 10, 20, 50, 100]:
        if n <= len(cumul):
            print(f"  Top {n:>3d} bairros: {cumul.iloc[n-1]:.2f}% dos registros ({cumul.index[n-1]})")

    # D6. Cruzamento bairro x natureza apurada (se existir)
    col_nat = None
    for c in df_sp.columns:
        if c.strip().upper() in ("NATUREZA_APURADA", "NATUREZA", "DESC_NATUREZA"):
            col_nat = c
            break
    if col_nat:
        print(f"\n  --- Top 10 bairros x Top 5 naturezas apuradas ---")
        top10_bairros = bairro_counts.head(10).index
        top5_nat = df_sp[col_nat].value_counts().head(5).index
        cross = pd.crosstab(df_sp[col_bairro], df_sp[col_nat])
        cross_top = cross.loc[cross.index.isin(top10_bairros), cross.columns.isin(top5_nat)]
        print(cross_top.to_string())

    # D7. Cruzamento bairro x mês (se existir coluna de data ou mês)
    col_data = None
    for c in df_sp.columns:
        if c.strip().upper() in ("DATA_OCORRENCIA", "DATAFATO", "DATA_FATO", "DATA", "MES", "Mês", "MES_OCORRENCIA"):
            col_data = c
            break
    if col_data is None:
        for c in df_sp.columns:
            if "DATA" in c.upper() or "MES" in c.upper() or "MÊS" in c.upper():
                col_data = c
                break
    if col_data:
        print(f"\n  --- Registros por mês (S.PAULO) ---")
        mes_counts = df_sp[col_data].value_counts().sort_index()
        print(f"  Coluna: '{col_data}'")
        print(mes_counts.to_string())

        # Top 10 bairros por mês (pivot)
        print(f"\n  --- Top 10 bairros x {col_data} ---")
        pivot_mes = pd.crosstab(df_sp[col_bairro], df_sp[col_data])
        pivot_top = pivot_mes.loc[pivot_mes.index.isin(bairro_counts.head(10).index)]
        print(pivot_top.to_string())

else:
    print(f"  Coluna de BAIRRO não encontrada.")
    print(f"  Colunas disponíveis: {list(df_sp.columns)}")

print(f"\n{'#' * 70}\n")


In [0]:
%pip install seaborn

# === Visualização EDA 2.5: SSP — SPDadosCriminais (2026) — S.PAULO ===
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

ssp_dir = "/Volumes/workspace/bronze/raw/ssp"
path_sp = os.path.join(ssp_dir, "SPDadosCriminais_2026.xlsx")

# ── Carregar e unificar as duas últimas abas (igual ao bloco 15) ──
xls_sp = pd.ExcelFile(path_sp)
ultimas_duas = xls_sp.sheet_names[-2:]
dfs = []
for aba in ultimas_duas:
    df_aba = pd.read_excel(path_sp, sheet_name=aba)
    dfs.append(df_aba)
df_unificado = pd.concat(dfs, ignore_index=True, sort=False)

# Identificar coluna de cidade
for c in df_unificado.columns:
    if c.strip().upper() in ("CIDADE", "MUNICIPIO", "MUNICÍPIO", "CIDADE_FATO", "NOME_MUNICIPIO", "NOME_MUNICÍPIO"):
        col_cidade = c
        break
if col_cidade is None:
    for c in df_unificado.columns:
        if "CIDADE" in c.upper() or "MUNICI" in c.upper():
            col_cidade = c
            break

# Filtrar S.PAULO
mask_sp = df_unificado[col_cidade].astype(str).str.strip().str.upper() == "S.PAULO"
df_sp = df_unificado[mask_sp].copy()

print(f"DataFrame S.PAULO: {df_sp.shape[0]:,} linhas x {df_sp.shape[1]} colunas")

# ── Identificar coluna de bairro ──────────────────────────────
col_bairro = None
for c in df_sp.columns:
    if c.strip().upper() in ("BAIRRO", "NOME_BAIRRO", "BAIRRO_FATO", "DESC_BAIRRO"):
        col_bairro = c
        break
if col_bairro is None:
    for c in df_sp.columns:
        if "BAIRRO" in c.upper():
            col_bairro = c
            break

# ── Identificar coluna de natureza apurada ───────────────────
col_nat = None
for c in df_sp.columns:
    if c.strip().upper() in ("NATUREZA_APURADA", "NATUREZA", "DESC_NATUREZA"):
        col_nat = c
        break

# ── Identificar coluna de data/mês ────────────────────────────
col_data = None
for c in df_sp.columns:
    if c.strip().upper() in ("DATA_OCORRENCIA", "DATAFATO", "DATA_FATO", "DATA", "MES", "Mês", "MES_OCORRENCIA"):
        col_data = c
        break
if col_data is None:
    for c in df_sp.columns:
        if "DATA" in c.upper() or "MES" in c.upper() or "MÊS" in c.upper():
            col_data = c
            break

# ══════════════════════════════════════════════════════════════
# 1. Valores ausentes por coluna
# ══════════════════════════════════════════════════════════════
miss = df_sp.isnull().sum()
miss = miss[miss > 0].sort_values(ascending=True)
miss_pct = (miss / len(df_sp) * 100)

if len(miss) > 0:
    fig, ax = plt.subplots(figsize=(14, max(4, len(miss) * 0.4)))
    ax.barh(miss.index, miss_pct.values, color="#C44E52")
    ax.set_xlabel("% de valores ausentes")
    ax.set_title("Valores Ausentes (%) — SPDadosCriminais (S.PAULO)", fontsize=13, pad=12)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}%"))
    for i, v in enumerate(miss_pct.values):
        ax.text(v + max(miss_pct.values) * 0.01, i, f"{v:.1f}%", va="center", fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum valor ausente.")

# ══════════════════════════════════════════════════════════════
# 2. Top 20 bairros por quantidade de registros
# ══════════════════════════════════════════════════════════════
if col_bairro:
    bairro_counts = df_sp[col_bairro].value_counts()
    top20 = bairro_counts.head(20)
    total = len(df_sp)

    fig, ax = plt.subplots(figsize=(14, 10))
    ax.barh(top20.index[::-1], top20.values[::-1], color="steelblue")
    ax.set_xlabel("Quantidade de registros")
    ax.set_title("Top 20 Bairros — Registros Criminais (S.PAULO)", fontsize=13, pad=12)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    for i, v in enumerate(top20.values[::-1]):
        pct = v / total * 100
        ax.text(v + max(top20.values) * 0.01, i, f"{v:,} ({pct:.1f}%)", va="center", fontsize=8)
    plt.tight_layout()
    plt.show()

    # ════════════════════════════════════════════════════════════
    # 3. Curva de Pareto — concentração nos top 30 bairros
    # ════════════════════════════════════════════════════════════
    cumul_pct = bairro_counts.cumsum() / bairro_counts.sum() * 100

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.bar(range(min(30, len(bairro_counts))), bairro_counts.values[:30], color="steelblue", alpha=0.8)
    ax.set_xlabel("Rank do bairro (top 30)")
    ax.set_ylabel("Quantidade de registros")
    ax.set_title("Distribuição de Registros por Bairro (Pareto) — S.PAULO", fontsize=13, pad=12)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

    ax2 = ax.twinx()
    ax2.plot(range(min(30, len(cumul_pct))), cumul_pct.values[:30], color="crimson", marker="o", linewidth=2)
    ax2.set_ylabel("% acumulada", color="crimson")
    ax2.set_ylim(0, 105)
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
    ax2.tick_params(axis="y", labelcolor="crimson")
    plt.tight_layout()
    plt.show()

    # ════════════════════════════════════════════════════════════
    # 4. Distribuição de registros por bairro (box plot + histograma)
    # ════════════════════════════════════════════════════════════
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Box plot
    bp = axes[0].boxplot(bairro_counts.values, vert=True, patch_artist=True, showfliers=True,
                        boxprops=dict(facecolor="steelblue", alpha=0.7),
                        flierprops=dict(marker="o", markerfacecolor="#C44E52", markersize=4, alpha=0.6))
    axes[0].set_ylabel("Registros por bairro")
    axes[0].set_title("Box Plot — Distribuição de Registros por Bairro", fontsize=12)
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

    # Histograma
    axes[1].hist(bairro_counts.values, bins=50, color="steelblue", alpha=0.8, edgecolor="white")
    axes[1].set_xlabel("Registros por bairro")
    axes[1].set_ylabel("Número de bairros")
    axes[1].set_title("Histograma — Registros por Bairro", fontsize=12)
    axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

    plt.tight_layout()
    plt.show()

    # ════════════════════════════════════════════════════════════
    # 5. Top 10 bairros x Top 10 naturezas apuradas (heatmap)
    # ════════════════════════════════════════════════════════════
    if col_nat:
        top10_bairros = bairro_counts.head(10).index
        top10_nat = df_sp[col_nat].value_counts().head(10).index
        cross = pd.crosstab(df_sp[col_bairro], df_sp[col_nat])
        cross_top = cross.loc[cross.index.isin(top10_bairros), cross.columns.isin(top10_nat)]

        fig, ax = plt.subplots(figsize=(16, 8))
        sns.heatmap(cross_top, annot=True, fmt="d", cmap="YlOrRd", ax=ax, cbar_kws={"label": "Registros"}, linewidths=0.5)
        ax.set_title("Heatmap — Top 10 Bairros x Top 10 Naturezas Apuradas (S.PAULO)", fontsize=13, pad=12)
        ax.set_xlabel("Natureza Apurada")
        ax.set_ylabel("Bairro")
        plt.xticks(rotation=45, ha="right", fontsize=9)
        plt.yticks(fontsize=9)
        plt.tight_layout()
        plt.show()

    # ════════════════════════════════════════════════════════════
    # 6. Registros por mês/coluna de data
    # ════════════════════════════════════════════════════════════
    if col_data:
        mes_counts = df_sp[col_data].value_counts().sort_index()

        fig, ax = plt.subplots(figsize=(14, 6))
        ax.bar(mes_counts.index.astype(str), mes_counts.values, color="steelblue")
        ax.set_title(f"Registros por {col_data} — S.PAULO", fontsize=13, pad=12)
        ax.set_ylabel("Quantidade de registros")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
        plt.xticks(rotation=45, ha="right", fontsize=9)
        for i, v in enumerate(mes_counts.values):
            ax.text(i, v + max(mes_counts.values) * 0.01, f"{v:,}", ha="center", fontsize=8)
        plt.tight_layout()
        plt.show()

        # Top 10 bairros x mês (heatmap)
        pivot_mes = pd.crosstab(df_sp[col_bairro], df_sp[col_data])
        pivot_top = pivot_mes.loc[pivot_mes.index.isin(bairro_counts.head(10).index)]

        fig, ax = plt.subplots(figsize=(16, 8))
        sns.heatmap(pivot_top, annot=True, fmt="d", cmap="Blues", ax=ax, cbar_kws={"label": "Registros"}, linewidths=0.5)
        ax.set_title(f"Heatmap — Top 10 Bairros x {col_data} (S.PAULO)", fontsize=13, pad=12)
        ax.set_xlabel(col_data)
        ax.set_ylabel("Bairro")
        plt.xticks(rotation=45, ha="right", fontsize=9)
        plt.yticks(fontsize=9)
        plt.tight_layout()
        plt.show()

    # ════════════════════════════════════════════════════════════
    # 7. Top 10 naturezas apuradas (bar chart horizontal)
    # ════════════════════════════════════════════════════════════
    if col_nat:
        nat_counts = df_sp[col_nat].value_counts().head(10)
        total_nat = len(df_sp)

        fig, ax = plt.subplots(figsize=(14, 8))
        ax.barh(nat_counts.index[::-1], nat_counts.values[::-1], color="#55A868")
        ax.set_xlabel("Quantidade de registros")
        ax.set_title("Top 10 Naturezas Apuradas — S.PAULO", fontsize=13, pad=12)
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
        for i, v in enumerate(nat_counts.values[::-1]):
            pct = v / total_nat * 100
            ax.text(v + max(nat_counts.values) * 0.01, i, f"{v:,} ({pct:.1f}%)", va="center", fontsize=8)
        plt.tight_layout()
        plt.show()
else:
    print("Coluna de BAIRRO não encontrada. Colunas disponíveis:")
    print(list(df_sp.columns))

print("\nVisualizações concluídas.")


In [0]:
# === EDA: SSP — VeiculosSubtraidos_2026 — Listar abas + EDA orientado por bairros (VEICULOS_2026) ===
import os
import pandas as pd

ssp_dir = "/Volumes/workspace/bronze/raw/ssp"
path_vs = os.path.join(ssp_dir, "VeiculosSubtraidos_2026.xlsx")

# ── 1. Verificar se o arquivo existe ──────────────────────────
if not os.path.exists(path_vs):
    print(f"⚠ Arquivo não encontrado: {path_vs}")
    print("Verifique o caminho e se o arquivo foi carregado no Volume.")
else:
    # ── 2. Carregar o arquivo e listar abas ────────────────────
    xls_vs = pd.ExcelFile(path_vs)
    print("=" * 70)
    print("SSP — VeiculosSubtraidos (2026)")
    print("=" * 70)
    print(f"Arquivo: {path_vs}")
    print(f"Tamanho: {os.path.getsize(path_vs) / (1024 * 1024):.1f} MB")
    print(f"\nAbas ({len(xls_vs.sheet_names)}):")
    for i, s in enumerate(xls_vs.sheet_names):
        print(f"  [{i}] {s}")

    # ── 3. Carregar a aba VEICULOS_2026 ────────────────────────
    aba_alvo = "VEICULOS_2026"
    if aba_alvo not in xls_vs.sheet_names:
        # Busca flexível (case-insensitive, sem acento/spaço)
        aba_alvo = None
        for s in xls_vs.sheet_names:
            if s.strip().upper().replace("_", "") == "VEICULOS2026":
                aba_alvo = s
                break
        if aba_alvo is None:
            print(f"\n⚠ Aba 'VEICULOS_2026' não encontrada.")
            print(f"  Abas disponíveis: {xls_vs.sheet_names}")
            raise SystemExit

    df = pd.read_excel(path_vs, sheet_name=aba_alvo)
    print(f"\n{'#' * 70}")
    print(f"# EDA — Aba: {aba_alvo}")
    print(f"{'#' * 70}")
    print(f"  Shape: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

    # (A) Colunas e tipos
    print(f"\n{'─' * 50}")
    print("(A) Colunas e tipos")
    print(f"{'─' * 50}")
    for c, t in df.dtypes.items():
        print(f"  {c:45s} {t}")

    # (B) Valores ausentes
    print(f"\n{'─' * 50}")
    print("(B) Taxa de valores nulos")
    print(f"{'─' * 50}")
    miss = df.isnull().sum()
    miss_pct = (miss / len(df) * 100).round(2)
    miss_df = pd.DataFrame({"nulos": miss, "taxa_null_%": miss_pct}).sort_values("taxa_null_%", ascending=False)
    has_null = miss_df[miss_df["nulos"] > 0]
    if len(has_null) > 0:
        for c, row in has_null.iterrows():
            print(f"  {c:45s} {row['nulos']:>8,}  ({row['taxa_null_%']:.2f}%)")
    else:
        print("  Nenhum valor nulo encontrado.")

    # (C) Estatísticas descritivas (colunas numéricas)
    print(f"\n{'─' * 50}")
    print("(C) Estatísticas descritivas (colunas numéricas)")
    print(f"{'─' * 50}")
    num_cols = df.select_dtypes(include=["number"]).columns
    if len(num_cols) > 0:
        print(df[num_cols].describe().round(2).to_string())
    else:
        print("  Nenhuma coluna numérica encontrada.")

    # (D) Amostra (primeiras 5 linhas)
    print(f"\n{'─' * 50}")
    print("(D) Amostra (primeiras 5 linhas)")
    print(f"{'─' * 50}")
    print(df.head().to_string(max_cols=15, max_colwidth=30))

    # ── 4. EDA orientado por bairros ────────────────────────────
    print(f"\n{'─' * 50}")
    print("(E) Análise por bairros")
    print(f"{'─' * 50}")

    # Identificar coluna de bairro
    col_bairro = None
    for c in df.columns:
        if c.strip().upper() in ("BAIRRO", "NOME_BAIRRO", "BAIRRO_FATO", "DESC_BAIRRO", "BAIRROOCORRENCIA", "BAIRRO_OCORRENCIA"):
            col_bairro = c
            break
    if col_bairro is None:
        for c in df.columns:
            if "BAIRRO" in c.upper():
                col_bairro = c
                break

    # Identificar coluna de cidade/município (para filtrar S.PAULO se existir)
    col_cidade = None
    for c in df.columns:
        if c.strip().upper() in ("CIDADE", "MUNICIPIO", "MUNICÍPIO", "CIDADE_FATO", "NOME_MUNICIPIO", "NOME_MUNICÍPIO"):
            col_cidade = c
            break
    if col_cidade is None:
        for c in df.columns:
            if "CIDADE" in c.upper() or "MUNICI" in c.upper():
                col_cidade = c
                break

    # Filtrar S.PAULO se houver coluna de cidade
    df_sp = df.copy()
    if col_cidade:
        print(f"\n  Coluna de cidade identificada: '{col_cidade}'")
        print(f"  Amostra de cidades (top 15 por frequência):")
        print(df[col_cidade].value_counts().head(15).to_string())
        mask_sp = df[col_cidade].astype(str).str.strip().str.upper() == "S.PAULO"
        n_sp = mask_sp.sum()
        print(f"\n  Registros para S.PAULO: {n_sp:,}")
        if n_sp > 0:
            df_sp = df[mask_sp].copy()
    else:
        print("\n  Coluna de cidade não encontrada — usando DataFrame completo.")

    print(f"\n  DataFrame para análise: {df_sp.shape[0]:,} linhas x {df_sp.shape[1]} colunas")

    if col_bairro:
        total = len(df_sp)
        bairro_counts = df_sp[col_bairro].value_counts()
        bairro_pct = (bairro_counts / total * 100).round(2)
        bairro_df = pd.DataFrame({"registros": bairro_counts, "%_do_total": bairro_pct})

        print(f"  Coluna identificada: '{col_bairro}'")
        print(f"  Total de bairros únicos: {bairro_counts.shape[0]}")
        print(f"  Total de registros: {total:,}")

        # E1. Top 20 bairros por quantidade de registros
        print(f"\n  --- Top 20 bairros por registros ---")
        print(f"  {'Bairro':<45s} {'Registros':>10s} {'% Total':>10s}")
        print(f"  {'-' * 45} {'-' * 10} {'-' * 10}")
        for bairro, row in bairro_df.head(20).iterrows():
            print(f"  {str(bairro):<45s} {row['registros']:>10,} {row['%_do_total']:>9.2f}%")
        print(f"  \n  Total Top 20:   {bairro_df.head(20)['registros'].sum():>10,}  ({bairro_df.head(20)['%_do_total'].sum():.2f}% do total)")

        # E2. Bottom 20 bairros (menos registros)
        print(f"\n  --- Bottom 20 bairros por registros ---")
        print(f"  {'Bairro':<45s} {'Registros':>10s} {'% Total':>10s}")
        print(f"  {'-' * 45} {'-' * 10} {'-' * 10}")
        for bairro, row in bairro_df.tail(20).iterrows():
            print(f"  {str(bairro):<45s} {row['registros']:>10,} {row['%_do_total']:>9.2f}%")

        # E3. Bairros nulos ou vazios
        nulos_bairro = df_sp[col_bairro].isnull().sum()
        vazios_bairro = (df_sp[col_bairro].astype(str).str.strip() == "").sum()
        print(f"\n  Registros com bairro nulo: {nulos_bairro:,} ({nulos_bairro/total*100:.2f}%)")
        print(f"  Registros com bairro vazio: {vazios_bairro:,} ({vazios_bairro/total*100:.2f}%)")

        # E4. Estatísticas da distribuição
        print(f"\n  --- Estatísticas da distribuição de registros por bairro ---")
        print(f"  Média de registros por bairro:    {bairro_counts.mean():,.1f}")
        print(f"  Mediana de registros por bairro: {bairro_counts.median():,.1f}")
        print(f"  Desvio padrão:                   {bairro_counts.std():,.1f}")
        print(f"  Mínimo:                          {bairro_counts.min():,}")
        print(f"  Máximo:                          {bairro_counts.max():,}")

        

    else:
        print(f"  Coluna de BAIRRO não encontrada.")
        print(f"  Colunas disponíveis: {list(df_sp.columns)}")

    print(f"\n{'=' * 70}")

In [0]:
%pip install seaborn

# === Visualização EDA: SSP — VeiculosSubtraidos_2026 (VEICULOS_2026) — S.PAULO ===
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

ssp_dir = "/Volumes/workspace/bronze/raw/ssp"
path_vs = os.path.join(ssp_dir, "VeiculosSubtraidos_2026.xlsx")

# ── Carregar a aba VEICULOS_2026 (igual ao bloco 17) ──────────
xls_vs = pd.ExcelFile(path_vs)
aba_alvo = "VEICULOS_2026"
if aba_alvo not in xls_vs.sheet_names:
    for s in xls_vs.sheet_names:
        if s.strip().upper().replace("_", "") == "VEICULOS2026":
            aba_alvo = s
            break

df = pd.read_excel(path_vs, sheet_name=aba_alvo)
print(f"DataFrame carregado: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

# ── Identificar coluna de bairro ──────────────────────────────
col_bairro = None
for c in df.columns:
    if c.strip().upper() in ("BAIRRO", "NOME_BAIRRO", "BAIRRO_FATO", "DESC_BAIRRO", "BAIRROOCORRENCIA", "BAIRRO_OCORRENCIA"):
        col_bairro = c
        break
if col_bairro is None:
    for c in df.columns:
        if "BAIRRO" in c.upper():
            col_bairro = c
            break

# ── Identificar coluna de cidade/município ────────────────────
col_cidade = None
for c in df.columns:
    if c.strip().upper() in ("CIDADE", "MUNICIPIO", "MUNICÍPIO", "CIDADE_FATO", "NOME_MUNICIPIO", "NOME_MUNICÍPIO"):
        col_cidade = c
        break
if col_cidade is None:
    for c in df.columns:
        if "CIDADE" in c.upper() or "MUNICI" in c.upper():
            col_cidade = c
            break

# ── Filtrar S.PAULO se houver coluna de cidade ────────────────
df_sp = df.copy()
if col_cidade:
    mask_sp = df[col_cidade].astype(str).str.strip().str.upper() == "S.PAULO"
    n_sp = mask_sp.sum()
    print(f"Registros S.PAULO: {n_sp:,}")
    if n_sp > 0:
        df_sp = df[mask_sp].copy()
else:
    print("Coluna de cidade não encontrada — usando DataFrame completo.")

print(f"DataFrame para análise: {df_sp.shape[0]:,} linhas x {df_sp.shape[1]} colunas")

# ── Identificar coluna de natureza/tipo ───────────────────────
col_nat = None
for c in df_sp.columns:
    if c.strip().upper() in ("NATUREZA_APURADA", "NATUREZA", "DESC_NATUREZA", "TIPO", "TIPO_CRIME", "TIPO_VEICULO"):
        col_nat = c
        break

# ── Identificar coluna de data/mês ────────────────────────────
col_data = None
for c in df_sp.columns:
    if c.strip().upper() in ("DATA_OCORRENCIA", "DATAFATO", "DATA_FATO", "DATA", "MES", "Mês", "MES_OCORRENCIA", "DATAOCORRENCIA"):
        col_data = c
        break
if col_data is None:
    for c in df_sp.columns:
        if "DATA" in c.upper() or "MES" in c.upper() or "MÊS" in c.upper():
            col_data = c
            break

if col_bairro is None:
    print("Coluna de BAIRRO não encontrada. Colunas disponíveis:")
    print(list(df_sp.columns))
    raise SystemExit

# ══════════════════════════════════════════════════════════════
# 1. Valores ausentes por coluna
# ══════════════════════════════════════════════════════════════
miss = df_sp.isnull().sum()
miss = miss[miss > 0].sort_values(ascending=True)
miss_pct = (miss / len(df_sp) * 100)

if len(miss) > 0:
    fig, ax = plt.subplots(figsize=(14, max(4, len(miss) * 0.4)))
    ax.barh(miss.index, miss_pct.values, color="#C44E52")
    ax.set_xlabel("% de valores ausentes")
    ax.set_title("Valores Ausentes (%) — VeiculosSubtraidos (S.PAULO)", fontsize=13, pad=12)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}%"))
    for i, v in enumerate(miss_pct.values):
        ax.text(v + max(miss_pct.values) * 0.01, i, f"{v:.1f}%", va="center", fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum valor ausente.")

# ══════════════════════════════════════════════════════════════
# 2. Top 20 bairros por quantidade de registros
# ══════════════════════════════════════════════════════════════
bairro_counts = df_sp[col_bairro].value_counts()
top20 = bairro_counts.head(20)
total = len(df_sp)

fig, ax = plt.subplots(figsize=(14, 10))
ax.barh(top20.index[::-1], top20.values[::-1], color="steelblue")
ax.set_xlabel("Quantidade de registros")
ax.set_title("Top 20 Bairros — Veículos Subtraídos (S.PAULO)", fontsize=13, pad=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
for i, v in enumerate(top20.values[::-1]):
    pct = v / total * 100
    ax.text(v + max(top20.values) * 0.01, i, f"{v:,} ({pct:.1f}%)", va="center", fontsize=8)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════
# 3. Bottom 20 bairros por quantidade de registros
# ══════════════════════════════════════════════════════════════
bottom20 = bairro_counts.tail(20)

fig, ax = plt.subplots(figsize=(14, 8))
ax.barh(bottom20.index[::-1], bottom20.values[::-1], color="#C44E52")
ax.set_xlabel("Quantidade de registros")
ax.set_title("Bottom 20 Bairros — Menos Registros (S.PAULO)", fontsize=13, pad=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
for i, v in enumerate(bottom20.values[::-1]):
    pct = v / total * 100
    ax.text(v + max(bottom20.values) * 0.01, i, f"{v:,} ({pct:.2f}%)", va="center", fontsize=8)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════
# 4. Curva de Pareto — concentração nos top 30 bairros
# ══════════════════════════════════════════════════════════════
cumul_pct = bairro_counts.cumsum() / bairro_counts.sum() * 100

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(range(min(30, len(bairro_counts))), bairro_counts.values[:30], color="steelblue", alpha=0.8)
ax.set_xlabel("Rank do bairro (top 30)")
ax.set_ylabel("Quantidade de registros")
ax.set_title("Distribuição de Registros por Bairro (Pareto) — Veículos Subtraídos", fontsize=13, pad=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

ax2 = ax.twinx()
ax2.plot(range(min(30, len(cumul_pct))), cumul_pct.values[:30], color="crimson", marker="o", linewidth=2)
ax2.set_ylabel("% acumulada", color="crimson")
ax2.set_ylim(0, 105)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax2.tick_params(axis="y", labelcolor="crimson")
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════
# 5. Distribuição de registros por bairro (box plot + histograma)
# ══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Box plot
axes[0].boxplot(bairro_counts.values, vert=True, patch_artist=True, showfliers=True,
                boxprops=dict(facecolor="steelblue", alpha=0.7),
                flierprops=dict(marker="o", markerfacecolor="#C44E52", markersize=4, alpha=0.6))
axes[0].set_ylabel("Registros por bairro")
axes[0].set_title("Box Plot — Distribuição de Registros por Bairro", fontsize=12)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Histograma
axes[1].hist(bairro_counts.values, bins=50, color="steelblue", alpha=0.8, edgecolor="white")
axes[1].set_xlabel("Registros por bairro")
axes[1].set_ylabel("Número de bairros")
axes[1].set_title("Histograma — Registros por Bairro", fontsize=12)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════
# 6. Top 10 bairros x Top 10 naturezas/tipos (heatmap)
# ══════════════════════════════════════════════════════════════
if col_nat:
    top10_bairros = bairro_counts.head(10).index
    top10_nat = df_sp[col_nat].value_counts().head(10).index
    cross = pd.crosstab(df_sp[col_bairro], df_sp[col_nat])
    cross_top = cross.loc[cross.index.isin(top10_bairros), cross.columns.isin(top10_nat)]

    fig, ax = plt.subplots(figsize=(16, 8))
    sns.heatmap(cross_top, annot=True, fmt="d", cmap="YlOrRd", ax=ax, cbar_kws={"label": "Registros"}, linewidths=0.5)
    ax.set_title(f"Heatmap — Top 10 Bairros x Top 10 {col_nat} (S.PAULO)", fontsize=13, pad=12)
    ax.set_xlabel(col_nat)
    ax.set_ylabel("Bairro")
    plt.xticks(rotation=45, ha="right", fontsize=9)
    plt.yticks(fontsize=9)
    plt.tight_layout()
    plt.show()

    # ════════════════════════════════════════════════════════════
    # 7. Top 10 naturezas/tipos (bar chart horizontal)
    # ══════════════════════════════════════════════════════════════
    nat_counts = df_sp[col_nat].value_counts().head(10)
    total_nat = len(df_sp)

    fig, ax = plt.subplots(figsize=(14, 8))
    ax.barh(nat_counts.index[::-1], nat_counts.values[::-1], color="#55A868")
    ax.set_xlabel("Quantidade de registros")
    ax.set_title(f"Top 10 {col_nat} — S.PAULO", fontsize=13, pad=12)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    for i, v in enumerate(nat_counts.values[::-1]):
        pct = v / total_nat * 100
        ax.text(v + max(nat_counts.values) * 0.01, i, f"{v:,} ({pct:.1f}%)", va="center", fontsize=8)
    plt.tight_layout()
    plt.show()

# ══════════════════════════════════════════════════════════════
# 8. Registros por mês/coluna de data
# ══════════════════════════════════════════════════════════════
if col_data:
    mes_counts = df_sp[col_data].value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.bar(mes_counts.index.astype(str), mes_counts.values, color="steelblue")
    ax.set_title(f"Registros por {col_data} — S.PAULO", fontsize=13, pad=12)
    ax.set_ylabel("Quantidade de registros")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    plt.xticks(rotation=45, ha="right", fontsize=9)
    for i, v in enumerate(mes_counts.values):
        ax.text(i, v + max(mes_counts.values) * 0.01, f"{v:,}", ha="center", fontsize=8)
    plt.tight_layout()
    plt.show()

    # Top 10 bairros x mês (heatmap)
    pivot_mes = pd.crosstab(df_sp[col_bairro], df_sp[col_data])
    pivot_top = pivot_mes.loc[pivot_mes.index.isin(bairro_counts.head(10).index)]

    fig, ax = plt.subplots(figsize=(16, 8))
    sns.heatmap(pivot_top, annot=True, fmt="d", cmap="Blues", ax=ax, cbar_kws={"label": "Registros"}, linewidths=0.5)
    ax.set_title(f"Heatmap — Top 10 Bairros x {col_data} (S.PAULO)", fontsize=13, pad=12)
    ax.set_xlabel(col_data)
    ax.set_ylabel("Bairro")
    plt.xticks(rotation=45, ha="right", fontsize=9)
    plt.yticks(fontsize=9)
    plt.tight_layout()
    plt.show()

# ══════════════════════════════════════════════════════════════
# 9. Estatísticas resumidas da distribuição por bairro
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("Estatísticas da distribuição de registros por bairro")
print("=" * 60)
print(f"  Total de bairros únicos:  {bairro_counts.shape[0]}")
print(f"  Total de registros:       {total:,}")
print(f"  Média por bairro:         {bairro_counts.mean():,.1f}")
print(f"  Mediana por bairro:      {bairro_counts.median():,.1f}")
print(f"  Desvio padrão:           {bairro_counts.std():,.1f}")
print(f"  Mínimo:                  {bairro_counts.min():,}")
print(f"  Máximo:                  {bairro_counts.max():,}")

nulos_bairro = df_sp[col_bairro].isnull().sum()
vazios_bairro = (df_sp[col_bairro].astype(str).str.strip() == "").sum()
print(f"  Bairros nulos:           {nulos_bairro:,} ({nulos_bairro/total*100:.2f}%)")
print(f"  Bairros vazios:          {vazios_bairro:,} ({vazios_bairro/total*100:.2f}%)")

# Gráfico de barras das estatísticas-chave
stats_labels = ["Mínimo", "Mediana", "Média", "Máximo"]
stats_values = [bairro_counts.min(), bairro_counts.median(), bairro_counts.mean(), bairro_counts.max()]

colors = ["#4C72B0", "#55A868", "#DD8452", "#C44E52"]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(stats_labels, stats_values, color=colors)
ax.set_ylabel("Registros")
ax.set_title("Estatísticas-Chave — Distribuição de Registros por Bairro", fontsize=13, pad=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
for i, v in enumerate(stats_values):
    ax.text(i, v + max(stats_values) * 0.01, f"{v:,.0f}", ha="center", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nVisualizações concluídas.")

In [0]:
# === EDA 3: ObservaSampa — Indicadores Socioeconômicos ===
import pandas as pd

df_obs = pd.read_csv("/Volumes/workspace/bronze/raw/observasampa/ObservaSampaDadosAbertosIndicadoresCSV.csv", encoding='utf-8-sig', sep=None, engine='python', on_bad_lines='skip')
df_obs = df_obs[df_obs["Período"] == df_obs["Período"].max()]

print("=" * 60)
print("ObservaSampa — Indicadores Socioeconômicos")
print("=" * 60)
print(f"Shape: {df_obs.shape[0]:,} linhas x {df_obs.shape[1]} colunas")
print(f"\nColunas: {list(df_obs.columns)}")
print(f"\nDtypes:")
for c, t in df_obs.dtypes.items():
    print(f"  {c:15s} {t}")

print(f"\nValores ausentes:")
miss = df_obs.isnull().sum()
miss = miss[miss > 0]
if len(miss) > 0:
    for c, v in miss.items():
        print(f"  {c:15s} {v:>6,} ({v/len(df_obs)*100:.1f}%)")
else:
    print("  Nenhum valor ausente")

print(f"\nIndicadores únicos: {df_obs['Nome'].nunique()}")
print(f"Regiões únicas:     {df_obs['Região'].nunique()}")
print(f"Período:            {df_obs['Período'].min()}–{df_obs['Período'].max()}")

print(f"\nEstatísticas de Resultado:")
print(df_obs['Resultado'].describe().to_string())

print(f"\nTop 10 indicadores por frequência:")
print(df_obs['Nome'].value_counts().head(10).to_string())

print(f"\nRegiões (amostra):")
print(df_obs['Região'].unique()[:15])

print(f"\nAmostra (primeiras 5 linhas):")
print(df_obs.head().to_string())

print(df_obs.describe().round(2).to_string())

In [0]:
# === EDA ObservaSampa — Análise Exploratória Completa e Visual ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

# ── Carregamento ─────────────────────────────────────────────
df_obs = pd.read_csv(
    "/Volumes/workspace/bronze/raw/observasampa/ObservaSampaDadosAbertosIndicadoresCSV.csv",
    encoding="utf-8-sig",
    sep=None,
    engine="python",
    on_bad_lines="skip",
)

print("=" * 70)
print("ObservaSampa — Indicadores Socioeconômicos de São Paulo")
print("=" * 70)
print(f"Shape total:       {df_obs.shape[0]:,} linhas x {df_obs.shape[1]} colunas")
print(f"Colunas:           {list(df_obs.columns)}")
print(f"Períodos:          {df_obs['Período'].min()} – {df_obs['Período'].max()} ({df_obs['Período'].nunique()} anos)")
print(f"Indicadores únicos: {df_obs['Nome'].nunique()}")
print(f"Regiões únicas:     {df_obs['Região'].nunique()}")

# Conversão segura de Resultado para numérico
df_obs["_resultado_num"] = pd.to_numeric(df_obs["Resultado"], errors="coerce")

# ── 1. Dtypes e valores ausentes ─────────────────────────────
print("\n--- Dtypes ---")
for c, t in df_obs.dtypes.items():
    print(f"  {c:20s} {t}")

print("\n--- Valores ausentes ---")
miss = df_obs.isnull().sum()
miss = miss[miss > 0].sort_values(ascending=False)
if len(miss) > 0:
    for c, v in miss.items():
        print(f"  {c:20s} {v:>8,} ({v / len(df_obs) * 100:.1f}%)")
else:
    print("  Nenhum valor ausente.")

# ── 2. Amostra de registros ──────────────────────────────────
print("\n--- Amostra de 10 registros ---")
cols_show = [c for c in ["Nome", "Período", "Região", "Resultado"] if c in df_obs.columns]
print(df_obs[cols_show].sample(10, random_state=42).to_string(index=False))
print()

# ── 3. Registros por período (todos os anos) ─────────────────
regs_por_periodo = df_obs.groupby("Período").size().sort_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(regs_por_periodo.index.astype(str), regs_por_periodo.values, color="#4C72B0")
ax.set_title("Quantidade de Registros por Período", fontsize=13, pad=12)
ax.set_ylabel("Quantidade de registros")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.xticks(rotation=45)
for i, v in enumerate(regs_por_periodo.values):
    ax.text(i, v + max(regs_por_periodo.values) * 0.01, f"{v:,}", ha="center", fontsize=8)
plt.tight_layout()
plt.show()


# ── 7. Trajetória temporal dos 6 indicadores mais frequentes ─
top6_nomes = df_obs["Nome"].value_counts().head(6).index

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, nm in zip(axes.ravel(), top6_nomes):
    sub = (
        df_obs[df_obs["Nome"] == nm]
        .dropna(subset=["_resultado_num"])
        .groupby("Período")["_resultado_num"]
        .median()
        .sort_index()
    )
    ax.plot(sub.index.astype(str), sub.values, marker="o", color="#4C72B0", linewidth=2)
    if len(sub) > 0:
        ax.fill_between(sub.index.astype(str), sub.values, alpha=0.15, color="#4C72B0")
    ax.set_title(nm, fontsize=10, pad=8)
    ax.set_xlabel("Período")
    ax.set_ylabel("Resultado (mediana)")
    ax.tick_params(axis="x", rotation=45, labelsize=8)
plt.suptitle("Trajetória Temporal — 6 Indicadores Mais Frequentes", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()



In [0]:
# === EDA 4: Imóveis — Preços por Bairro (Guru dos Imóveis) ===
import pandas as pd

df_imov = pd.read_csv("/Volumes/workspace/bronze/raw/imoveis/gurudosimoveis.csv", skiprows=1)

print("=" * 60)
print("Imóveis — Guru dos Imóveis (São Paulo/SP)")
print("=" * 60)
print(f"Shape: {df_imov.shape[0]:,} linhas x {df_imov.shape[1]} colunas")
print(f"\nColunas: {list(df_imov.columns)}")
print(f"\nDtypes:")
for c, t in df_imov.dtypes.items():
    print(f"  {c:30s} {t}")

print(f"\nValores ausentes:")
miss = df_imov.isnull().sum()
miss = miss[miss > 0]
if len(miss) > 0:
    for c, v in miss.items():
        print(f"  {c:30s} {v:>6,} ({v/len(df_imov)*100:.1f}%)")
else:
    print("  Nenhum valor ausente")

print(f"\nMeses:              {df_imov['mes'].nunique()} — {', '.join(df_imov['mes'].unique())}")
print(f"Bairros:            {df_imov['bairro'].nunique()}")
print(f"Tipos de transação: {', '.join(df_imov['transacao'].unique())}")

print(f"\nEstatísticas descritivas:")
print(df_imov.describe().round(2).to_string())

print(f"\nTop 10 bairros por mediana_m2 (venda):")
venda = df_imov[df_imov['transacao'] == 'venda'].sort_values('mediana_m2', ascending=False)
print(venda[['bairro','mediana_m2','p25_m2','p75_m2']].head(10).to_string(index=False))

print(f"\nTop 10 bairros por mediana_m2 (aluguel):")
alug = df_imov[df_imov['transacao'] == 'aluguel'].sort_values('mediana_m2', ascending=False)
print(alug[['bairro','mediana_m2','p25_m2','p75_m2']].head(10).to_string(index=False))

print(f"\nAmostra (primeiras 5 linhas):")
print(df_imov.head().to_string())

In [0]:
# === Visualização EDA 4: Imóveis — Preços por Bairro (Guru dos Imóveis) ===
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

df_imov = pd.read_csv("/Volumes/workspace/bronze/raw/imoveis/gurudosimoveis.csv", skiprows=1)

# ── 1. Top 15 bairros por mediana_m2 (venda) ─────────────────

# 1. Filtrar imóveis para transação de venda
df_venda = df_imov[
    (df_imov["transacao"] == "venda") &
    (df_imov["bairro"].notna()) &
    (df_imov["mediana_m2"].notna())
].copy()

# 2. Garantir que o preço seja numérico
df_venda["mediana_m2"] = pd.to_numeric(
    df_venda["mediana_m2"], errors="coerce"
)
df_venda = df_venda.dropna(subset=["mediana_m2"])

# 3. Sumarizar por bairro
venda = (
    df_venda
    .groupby("bairro", as_index=False)
    .agg(
        mediana_m2=("mediana_m2", "median"),
        quantidade_registros=("mediana_m2", "size")
    )
    .sort_values("mediana_m2", ascending=False)
    .head(15)
    .reset_index(drop=True)
)

# 4. Exibir os 15 bairros sumarizados
display(venda)

# 5. Construir o gráfico
venda_plot = venda.sort_values("mediana_m2", ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))

barras = ax.barh(
    venda_plot["bairro"],
    venda_plot["mediana_m2"],
    color="steelblue"
)

ax.set_title("Top 15 Bairros — Mediana do Preço de Venda por m²", pad=14)
ax.set_xlabel("Mediana do preço (R$/m²)")
ax.set_ylabel("Bairro")

# Formatação monetária brasileira
ax.xaxis.set_major_formatter(
    mticker.FuncFormatter(
        lambda x, _: f"R$ {x:,.0f}".replace(",", ".")
    )
)

# Valores ao lado das barras
margem = venda_plot["mediana_m2"].max() * 0.01

for barra, valor in zip(barras, venda_plot["mediana_m2"]):
    ax.text(
        valor + margem,
        barra.get_y() + barra.get_height() / 2,
        f"R$ {valor:,.0f}".replace(",", "."),
        va="center",
        fontsize=8
    )

ax.set_xlim(0, venda_plot["mediana_m2"].max() * 1.25)
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)

fig.tight_layout()
plt.show()



# ── 2. Top 15 bairros por mediana_m2 (aluguel) ───────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# 1. Filtrar imóveis para transação de aluguel
df_aluguel = df_imov[
    (df_imov["transacao"] == "aluguel") &
    (df_imov["bairro"].notna()) &
    (df_imov["mediana_m2"].notna())
].copy()

# 2. Garantir que o preço seja numérico
df_aluguel["mediana_m2"] = pd.to_numeric(
    df_aluguel["mediana_m2"], errors="coerce"
)
df_aluguel = df_aluguel.dropna(subset=["mediana_m2"])

# 3. Sumarizar por bairro
alug = (
    df_aluguel
    .groupby("bairro", as_index=False)
    .agg(
        mediana_m2=("mediana_m2", "median"),
        quantidade_registros=("mediana_m2", "size")
    )
    .sort_values("mediana_m2", ascending=False)
    .head(15)
    .reset_index(drop=True)
)

# 4. Exibir os 15 bairros sumarizados
display(alug)

# 5. Ordenar para visualização do gráfico
alug_plot = alug.sort_values("mediana_m2", ascending=True)

# 6. Criar gráfico horizontal
fig, ax = plt.subplots(figsize=(12, 8))

barras = ax.barh(
    alug_plot["bairro"],
    alug_plot["mediana_m2"],
    color="#DD8452"
)

ax.set_xlabel("Mediana do preço de aluguel (R$/m²)")
ax.set_ylabel("Bairro")
ax.set_title(
    "Top 15 Bairros — Mediana do Preço de Aluguel por m²",
    pad=14
)

# Formatação monetária brasileira
ax.xaxis.set_major_formatter(
    mticker.FuncFormatter(
        lambda x, _: f"R$ {x:,.0f}".replace(",", ".")
    )
)

# 7. Adicionar os valores ao lado das barras
margem = alug_plot["mediana_m2"].max() * 0.01

for barra, valor in zip(barras, alug_plot["mediana_m2"]):
    ax.text(
        valor + margem,
        barra.get_y() + barra.get_height() / 2,
        f"R$ {valor:,.0f}".replace(",", "."),
        va="center",
        fontsize=8
    )

# Reservar espaço para os rótulos
ax.set_xlim(0, alug_plot["mediana_m2"].max() * 1.25)

ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)

fig.tight_layout()
plt.show()


# ── 4. Boxplot comparativo venda vs aluguel ──────────────────
# ── 4a. Boxplot — Venda ────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
bp = ax.boxplot(venda_vals, labels=["Venda"], patch_artist=True, showfliers=True)
bp["boxes"][0].set_facecolor("steelblue")
ax.set_title("Distribuição Mediana R$/m² — Venda")
ax.set_ylabel("R$/m²")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R$ {x:,.0f}"))
plt.tight_layout()
plt.show()

# ── 4b. Boxplot — Aluguel ───────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
bp = ax.boxplot(alug_vals, labels=["Aluguel"], patch_artist=True, showfliers=True)
bp["boxes"][0].set_facecolor("#DD8452")
ax.set_title("Distribuição Mediana R$/m² — Aluguel")
ax.set_ylabel("R$/m²")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R$ {x:,.0f}"))
plt.tight_layout()
plt.show()



In [0]:
# === EDA 5: Geosampa — Infraestrutura por Distrito ===
import pandas as pd

df_geo = pd.read_csv("/Volumes/workspace/bronze/raw/geosampa/base_bairroiq_consolidada.csv", sep=None, engine='python')

print("=" * 60)
print("Geosampa — Infraestrutura por Distrito (BairroIQ)")
print("=" * 60)
print(f"Shape: {df_geo.shape[0]:,} linhas x {df_geo.shape[1]} colunas")
print(f"\nColunas: {list(df_geo.columns)}")
print(f"\nDtypes:")
for c, t in df_geo.dtypes.items():
    print(f"  {c:30s} {t}")

print(f"\nValores ausentes:")
miss = df_geo.isnull().sum()
miss = miss[miss > 0]
if len(miss) > 0:
    for c, v in miss.items():
        print(f"  {c:30s} {v}")
else:
    print("  Nenhum valor ausente")

print(f"\nDistritos: {df_geo['nm_distrit'].nunique()}")

# Total de infraestrutura por distrito
cols_infra = [c for c in df_geo.columns if c != 'nm_distrit']
df_geo['_total'] = df_geo[cols_infra].sum(axis=1)

print(f"\nTop 10 distritos por total de infraestrutura:")
top10 = df_geo[['nm_distrit','_total']].sort_values('_total', ascending=False).head(10)
print(top10.to_string(index=False))

print(f"\nBottom 10 distritos por total de infraestrutura:")
bot10 = df_geo[['nm_distrit','_total']].sort_values('_total').head(10)
print(bot10.to_string(index=False))

print(f"\nEstatísticas descritivas (infraestrutura):")
print(df_geo[cols_infra].describe().to_string())

print(f"\nAmostra (primeiras 5 linhas, colunas selecionadas):")
show_cols = ['nm_distrit','qtd_escolas_publicas','qtd_hospitais','qtd_ubs','qtd_metro','qtd_onibus','qtd_policia_militar']
print(df_geo[show_cols].head().to_string(index=False))

df_geo.drop(columns=['_total'], inplace=True)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração de estilo para deixar os gráficos mais bonitos
sns.set_theme(style="whitegrid")

# Recalculando o _total caso tenha sido dropado
cols_infra = df_geo.select_dtypes(include='number').columns.tolist()
df_geo['_total'] = df_geo[cols_infra].sum(axis=1)
top10 = df_geo[['nm_distrit','_total']].sort_values('_total', ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x='_total', y='nm_distrit', data=top10, hue='nm_distrit', palette='viridis', legend=False)

plt.title('Top 10 Distritos por Volume de Infraestrutura', fontsize=14, pad=15)
plt.xlabel('Total de Equipamentos (Escolas, Hospitais, etc.)', fontsize=12)
plt.ylabel('Distrito', fontsize=12)
plt.tight_layout()
plt.show()

In [0]:
# Filtra os 10 distritos com mais escolas públicas e ordena
top10_escolas = df_geo[['nm_distrit', 'qtd_escolas_publicas']].sort_values(
    by='qtd_escolas_publicas', 
    ascending=False
).head(10)

plt.figure(figsize=(10, 6))

# Cria o gráfico de barras horizontais
sns.barplot(
    x='qtd_escolas_publicas', 
    y='nm_distrit', 
    data=top10_escolas, 
    hue='nm_distrit', 
    palette='Blues_r', 
    legend=False
)

# Estilização
plt.title('Top 10 Distritos de São Paulo com Mais Escolas Públicas', fontsize=14, pad=15)
plt.xlabel('Quantidade de Escolas Públicas', fontsize=12)
plt.ylabel('Distrito', fontsize=12)
plt.tight_layout()

# Exibe o gráfico
plt.show()

In [0]:
# Filtra os 10 distritos com menos escolas públicas e ordena
top10_escolas = df_geo[['nm_distrit', 'qtd_escolas_publicas']].sort_values(
    by='qtd_escolas_publicas', 
    ascending=True
).head(10)

plt.figure(figsize=(10, 6))

# Cria o gráfico de barras horizontais
sns.barplot(
    x='qtd_escolas_publicas', 
    y='nm_distrit', 
    data=top10_escolas, 
    hue='nm_distrit', 
    palette='Blues_r', 
    legend=False
)

# Estilização
plt.title('Top 10 Distritos de São Paulo com menos Escolas Públicas', fontsize=14, pad=15)
plt.xlabel('Quantidade de Escolas Públicas', fontsize=12)
plt.ylabel('Distrito', fontsize=12)
plt.tight_layout()

# Exibe o gráfico
plt.show()

In [0]:
# === Heatmaps de Correlação por Base de Dados ===
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from zipfile import ZipFile

sns.set_theme(style="whitegrid")

# ------------------------------------------------------------------
# 1. IBGE — Correlação entre variáveis demográficas
# ------------------------------------------------------------------
zf = ZipFile("/Volumes/workspace/bronze/raw/ibge/agregados_basicos.zip")
with zf.open("Agregados_por_distritos_basico_BR.csv") as f:
    df_ibge = pd.read_csv(f, encoding='latin-1', sep=';', na_values=['.'], decimal=',')

num_ibge = ['v0001','v0002','v0003','v0004','v0005','v0006','v0007','v0008','v0009','AREA_KM2']
labels_ibge = {
    'v0001':'Pop. Total','v0002':'Pop. Homens','v0003':'Pop. Mulheres',
    'v0004':'Densidade','v0005':'Média Moradores/domicílio','v0006':'Densidade Domiciliar',
    'v0007':'Domicílios','v0008':'Domicílios Ocupados','v0009':'Domicílios Fechados',
    'AREA_KM2':'Área km²'
}
corr_ibge = df_ibge[num_ibge].rename(columns=labels_ibge).corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_ibge, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('IBGE — Correlação entre Variáveis Demográficas', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# 2. SSP — Correlação entre variáveis numéricas (MDIP)
# ------------------------------------------------------------------
df_ssp = pd.read_excel("/Volumes/workspace/bronze/raw/ssp/MDIP_2026.xlsx", sheet_name="MDIP_2013_A_2026")
df_ssp['IDADE_NUM'] = pd.to_numeric(df_ssp['IDADE_PESSOA'], errors='coerce')
df_ssp['LAT_NUM'] = pd.to_numeric(df_ssp['LATITUDE'], errors='coerce')
df_ssp['LONG_NUM'] = pd.to_numeric(df_ssp['LONGITUDE'], errors='coerce')
num_ssp = ['ANO ESTATISTICA','MÊS ESTATISTICA','IDADE_NUM','LAT_NUM','LONG_NUM']
labels_ssp = {'ANO ESTATISTICA':'Ano','MÊS ESTATISTICA':'Mês','IDADE_NUM':'Idade',
              'LAT_NUM':'Latitude','LONG_NUM':'Longitude'}
corr_ssp = df_ssp[num_ssp].rename(columns=labels_ssp).corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr_ssp, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('SSP (MDIP) — Correlação entre Variáveis Numéricas', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# 3. ObservaSampa — Heatmap de valores: Regiões × Indicadores (2025)
#    (Apenas 2 colunas numéricas — correlação não é viável.
#     Em vez disso, heatmap de valores por região e indicador.)
# ------------------------------------------------------------------
df_obs = pd.read_csv("/Volumes/workspace/bronze/raw/observasampa/ObservaSampaDadosAbertosIndicadoresCSV.csv",
                     encoding='utf-8-sig', sep=None, engine='python', on_bad_lines='skip')
df_obs['Resultado_num'] = pd.to_numeric(df_obs['Resultado'], errors='coerce')
df_obs_latest = df_obs[df_obs['Período'] == df_obs['Período'].max()]

# Selecionar os 15 indicadores com maior variância entre regiões
top_ind = df_obs_latest.groupby('Nome')['Resultado_num'].var().nlargest(15).index
pivot_obs = df_obs_latest[df_obs_latest['Nome'].isin(top_ind)].pivot_table(
    index='Região', columns='Nome', values='Resultado_num', aggfunc='first'
)
# Encurtar nomes dos indicadores para o eixo X
pivot_obs.columns = [c[:35] for c in pivot_obs.columns]

plt.figure(figsize=(14, 12))
sns.heatmap(pivot_obs, cmap='YlOrRd', linewidths=0.3, linecolor='gray',
            cbar_kws={'label': 'Resultado'})
plt.title(f'ObservaSampa — Top 15 Indicadores por Região ({df_obs["Período"].max()})',
         fontsize=14, pad=15)
plt.xlabel('Indicador')
plt.ylabel('Região')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# 4. Imóveis — Correlação entre variáveis numéricas
# ------------------------------------------------------------------
df_imov = pd.read_csv("/Volumes/workspace/bronze/raw/imoveis/gurudosimoveis.csv", skiprows=1)
num_imov = ['mediana_m2','p25_m2','p75_m2','anuncios_ativos_media_dia','novos_anuncios','dias_no_mercado_medio']
labels_imov = {
    'mediana_m2':'Mediana m²','p25_m2':'P25 m²','p75_m2':'P75 m²',
    'anuncios_ativos_media_dia':'Anúncios Ativos/dia','novos_anuncios':'Novos Anúncios',
    'dias_no_mercado_medio':'Dias no Mercado'
}
corr_imov = df_imov[num_imov].rename(columns=labels_imov).corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_imov, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Imóveis — Correlação entre Variáveis', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# 5. Geosampa — Correlação entre categorias de infraestrutura
#    (42 colunas individuais — agrupadas em 7 categorias para legibilidade)
# ------------------------------------------------------------------
df_geo = pd.read_csv("/Volumes/workspace/bronze/raw/geosampa/base_bairroiq_consolidada.csv", sep=None, engine='python')

categorias = {
    'Educação':      ['qtd_escolas_publicas','qtd_escolas_privadas','qtd_ensino_tecnico_publico','qtd_ensino_infatil_publico','qtd_sistema_s','qtd_ceu'],
    'Saúde':         ['qtd_hospitais','qtd_ubs','qtd_ambulatorios','qtd_saude_mental','qtd_urgencia_emergencia'],
    'Segurança':     ['qtd_policia_militar','qtd_policia_civil','qtd_guarda_civil','qtd_bombeiros','qtd_defesa_civil'],
    'Transporte':    ['qtd_metro','qtd_trem','qtd_onibus','qtd_terminal_onibus','qtd_bicicletario'],
    'Cultura/Lazer': ['qtd_bibliotecas','qtd_museus','qtd_teatros_cinemas','qtd_espacos','qtd_clubes','qtd_estadios','qtd_praca_largo','qtd_corredor_verde','qtd_centro_esportivo','qtd_clubes_comunidade'],
    'Serviços':      ['qtd_correios','qtd_feira_livre','qtd_mercado_municipal','qtd_eletropaulo','qtd_sabesp','qtd_pupatempo','qtd_sacoloes','qtd_consulados'],
    'Outros':        ['qtd_agencia_funeraria','qtd_velorio','qtd_cemiterio']
}

for cat, cols in categorias.items():
    df_geo[cat] = df_geo[cols].sum(axis=1)

corr_geo = df_geo[list(categorias.keys())].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_geo, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Geosampa — Correlação entre Categorias de Infraestrutura', fontsize=14, pad=15)
plt.tight_layout()
plt.show()